# Confidence and Density Rejection Under Distribution Shift in sEMG: full reproduction (V11)

This notebook regenerates every table and figure in manuscript V11 from the two public datasets. It downloads the data, parses it, runs all experiments, and saves the results, tables and figures.

**How to run.** Runtime > Run all. It runs on a CPU or a GPU runtime without changes.

**Runtime.** About 2 hours on a standard Colab CPU and well under an hour on a GPU. Set `QUICK = True` in the first cell for a short smoke test (about 15 to 20 minutes) that uses fewer seeds and epochs; its numbers will not match the paper.

**Disconnections.** With `SAVE_TO_DRIVE = True`, results go to Google Drive and every experiment skips the runs it has already finished, so after a disconnect you simply run all again and it resumes.

**GPU note.** A GPU run reproduces the findings but not every third decimal, because reduction order differs between processors. The paper's numbers were produced on CPU. TF32 is disabled so GPU arithmetic stays in full float32.

| Section | What it produces |
|---|---|
| 1 to 3 | Settings, data download and parsing |
| 4 | Shared code: encoder, probe, density gate, statistics, conditions, faults |
| 5 | Experiments: Dataset A ten seeds; within Dataset B ten splits; ten participant splits with depth; held-out gestures; 200 ms windows; classical LDA; within-participant control; participant-clustered intervals |
| 6 | Aggregation, Tables III to XII as CSV, Figures 2 to 10, download |


In [ ]:
# ---- 1. Settings ---------------------------------------------------------------
QUICK = False          # True: short smoke test, numbers will not match the paper
SAVE_TO_DRIVE = True   # keep results on Google Drive so the run can resume after a disconnect

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/emg_v10" + ("_quick" if QUICK else "")
else:
    WORK = "/content/emg_v10" + ("_quick" if QUICK else "")
os.makedirs("/content/emg_code", exist_ok=True)
for d in ["data", "raw", "results"]:
    os.makedirs(os.path.join(WORK, d), exist_ok=True)
os.environ["EMG_DATA"] = os.path.join(WORK, "data")
os.environ["EMG_RES"] = os.path.join(WORK, "results")
os.environ["PYTHONUNBUFFERED"] = "1"
if QUICK:
    os.environ.update(EPOCHS="3", SEEDS_A="42,43", SEEDS_B="0,1", NS="2", SEEDS_N="42", NPAIRS="2",
                      SEEDS_W="42", SEEDS_WP="42", SEEDS_CI="42,43", NBOOT="100")
else:
    os.environ.update(EPOCHS="25", SEEDS_A="42,43,44,45,46,47,48,49,50,51", SEEDS_B="0,1,2,3,4,5,6,7,8,9", NS="10",
                      SEEDS_N="42,43,44,45,46", NPAIRS="4", SEEDS_W="42,43,44,45,46", SEEDS_WP="42,43,44",
                      SEEDS_CI="42,43,44,45,46,47,48,49,50,51", NBOOT="1000")
print("working folder:", WORK)

def run(script, *args, **extra):
    """Run one experiment script with live output; extra keyword arguments become environment variables."""
    env = dict(os.environ, **{k: str(v) for k, v in extra.items()})
    p = subprocess.Popen([sys.executable, "-u", script, *args], cwd="/content/emg_code", env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(script + " failed; see the output above")


In [ ]:
# ---- 2. Environment -----------------------------------------------------------------
import importlib
for pkg in ["torch", "sklearn", "scipy", "pandas", "matplotlib"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg if pkg != "sklearn" else "scikit-learn"], check=True)
import torch, numpy, sklearn, scipy
print("torch", torch.__version__, "| numpy", numpy.__version__, "| scikit-learn", sklearn.__version__, "| scipy", scipy.__version__)
print("device:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU, " + str(os.cpu_count()) + " cores")


In [ ]:
# ---- 3. Download and parse both datasets --------------------------------------------
# Dataset A: Hu and Hu, Forearm sEMG gesture dataset (CC BY 4.0), GitHub Hunter-qing
# Dataset B: Francic, Myo readings dataset (MIT), GitHub aljazfrancic
import glob, zipfile
RAW = os.path.join(WORK, "raw")
repos = {"A": "https://github.com/Hunter-qing/Forearm-sEMG-Gesture-Dataset.git",
         "B": "https://github.com/aljazfrancic/myo-readings-dataset.git"}
for k, url in repos.items():
    dst = os.path.join(RAW, k)
    if not os.path.isdir(dst):
        subprocess.run(["git", "clone", "-q", "--depth", "1", url, dst], check=True)
A_ROOT = os.path.join(RAW, "A_extracted")
if not glob.glob(os.path.join(A_ROOT, "**", "subject_*"), recursive=True):
    with zipfile.ZipFile(os.path.join(RAW, "A", "dataset.zip")) as z:
        z.extractall(A_ROOT)
subj = sorted(glob.glob(os.path.join(A_ROOT, "**", "subject_*"), recursive=True))
A_SUBJ_PARENT = os.path.dirname(subj[0])
print("Dataset A subject folders:", len(subj), "under", A_SUBJ_PARENT)


In [ ]:
%%writefile /content/emg_code/prepare_data.py
"""Parse both public datasets into d1.npz (Dataset A) and d2.npz (Dataset B).
Usage: python prepare_data.py <dataset_A_root_with_subject_dirs> <dataset_B_repo_root>"""
import os, re, sys, glob, numpy as np, scipy.io as sio, pandas as pd

OUT = os.environ.get("EMG_DATA", os.path.dirname(os.path.abspath(__file__)))
N_T, N_CH, N_CLASS = 1000, 8, 6

# ---- Dataset A: Hunter-qing Forearm sEMG Gesture Dataset (CC BY 4.0), stored rectified
def _parse_name(path):
    m = re.search(r"subject_(\d+)/session_(\d+)/(.+)\.mat$", path.replace("\\", "/"))
    subj, sess, name = int(m.group(1)), int(m.group(2)), m.group(3)
    g = re.match(r"gesture_(\d+)_(\d+)", name)
    if g: return subj, sess, int(g.group(1))
    r = re.match(r"(?:rest|relax)[_-]?(\d+)", name, re.I)
    return subj, sess, (0 if r else None)

def load_a(root):
    files = sorted(glob.glob(os.path.join(root, "subject_*", "session_*", "*.mat")))
    X, y, subj, sess = [], [], [], []
    for f in files:
        s, ss, lab = _parse_name(f)
        d = sio.loadmat(f); emg = np.asarray(d["emg"], dtype=np.float32)
        if emg.shape != (N_T, N_CH): continue
        L = np.unique(np.asarray(d["label"]).ravel())
        if L.size == 1: lab = int(L[0])
        if lab is None or not (0 <= lab < N_CLASS): continue
        X.append(emg); y.append(lab); subj.append(s); sess.append(ss)
    return np.stack(X), np.asarray(y, np.int64), np.asarray(subj, np.int64), np.asarray(sess, np.int64)

# ---- Dataset B: aljazfrancic myo-readings-dataset (MIT), signed bytes, continuous files
D2_MAP = {0: 0, 7: 1, 3: 2, 4: 2}     # rest, fist, radial/ulnar deviation -> wrist deviation
MAX_WIN_PER_LABEL = 2

def parse_b_file(path):
    try:
        a = pd.read_csv(path, header=None, dtype=np.int16).values
    except Exception:
        return []
    if a.shape[1] != 9 or len(a) < N_T: return []
    emg, lab = a[:, :8].astype(np.float32), a[:, 8]          # cast before any absolute value
    out, i, taken, n = [], 0, {}, len(lab)
    while i < n - N_T:
        L = lab[i]
        if L not in D2_MAP: i += N_T // 2; continue
        if (lab[i:i + N_T] == L).all():                     # window must not cross a label change
            m = D2_MAP[L]
            if taken.get(m, 0) < MAX_WIN_PER_LABEL:
                out.append((emg[i:i + N_T], m)); taken[m] = taken.get(m, 0) + 1
            i += N_T
        else:
            i += N_T // 4
    return out

def load_b(root):
    X, y, subj, sess = [], [], [], []
    for sd in sorted(glob.glob(os.path.join(root, "_readings_right_hand", "*-*"))):
        bn = os.path.basename(sd)
        try: pid, sn = int(bn.split("-")[0]), int(bn.split("-")[1])
        except Exception: continue
        for f in sorted(glob.glob(os.path.join(sd, "*.txt"))):
            for w, m in parse_b_file(f):
                X.append(w); y.append(m); subj.append(pid); sess.append(sn)
    return np.stack(X), np.asarray(y, np.int64), np.asarray(subj, np.int64), np.asarray(sess, np.int64)

if __name__ == "__main__":
    os.makedirs(OUT, exist_ok=True)
    X, y, s, ss = load_a(sys.argv[1])
    np.savez_compressed(os.path.join(OUT, "d1.npz"), X=X, y=y, subj=s, sess=ss)
    print(f"Dataset A: {X.shape}, range {X.min():.0f} to {X.max():.0f}, class counts {np.bincount(y).tolist()}")
    X, y, s, ss = load_b(sys.argv[2])
    np.savez_compressed(os.path.join(OUT, "d2.npz"), X=X, y=y, subj=s, sess=ss)
    print(f"Dataset B: {X.shape}, range {X.min():.0f} to {X.max():.0f}, participants {len(np.unique(s))}, "
          f"class counts {np.bincount(y).tolist()}")


In [ ]:
if not (os.path.exists(os.path.join(WORK, "data", "d1.npz")) and os.path.exists(os.path.join(WORK, "data", "d2.npz"))):
    run("prepare_data.py", A_SUBJ_PARENT, os.path.join(RAW, "B"))
else:
    print("parsed data already present")
# Expected: Dataset A (3000, 1000, 8), range 0 to 128; Dataset B (1082, 1000, 8), range -128 to 127, 38 participants


## 4. Shared code
Encoder, pretraining, calibrated linear probe, Mahalanobis density gate, abstention statistics, sensor faults, conditioning mismatches and the classical time-domain baseline.

In [ ]:
%%writefile /content/emg_code/core.py
import os, sys, time, json, math, glob, re, warnings
import numpy as np, torch
warnings.filterwarnings("ignore")
DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
if DEV == "cpu": torch.set_num_threads(max(1, int(os.environ.get("EMG_THREADS", os.cpu_count() or 2))))
DATA = os.environ.get("EMG_DATA", os.path.dirname(os.path.abspath(__file__)))
EPOCHS=int(os.environ.get("EPOCHS",25)); BATCH=64
PATCH, N_TOK, D_MODEL, N_LAYER, N_HEAD = 20, 50, 96, 3, 4
PATCH_S, N_TOK_S = 5, 8
MASK_RATIO=0.40; DELTA=1.5; COV_RIDGE=1e-2; OOD_PCT=95.0
CALIB_SUBJECTS=[17,18,19,20]
N_T, N_CH, K6, KC = 1000, 8, 6, 3
D1_MAP = {0: 0, 1: 1, 3: 2, 4: 2}
def seed_all(s):
    import random; random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
def norm_fit(X):
    return X.mean((0,1), keepdims=True), X.std((0,1), keepdims=True) + 1e-6
# ===== CELL 8 =====
import torch.nn as nn

class EMGEncoder(nn.Module):
    def __init__(self, d=D_MODEL, n_layer=N_LAYER, n_head=N_HEAD, n_ch=N_CH,
                 n_tok=N_TOK, patch=PATCH):
        super().__init__()
        self.n_tok = n_tok
        self.pd = patch * n_ch
        self.proj = nn.Linear(self.pd, d)
        self.pos = nn.Parameter(torch.randn(1, n_tok, d) * 0.02)
        self.mask_tok = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, n_head, 4*d, dropout=0.1, batch_first=True,
                                           norm_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(layer, n_layer)
        self.norm = nn.LayerNorm(d); self.head = nn.Linear(d, self.pd)
    def patchify(self, x): return x.reshape(x.shape[0], self.n_tok, self.pd)
    def forward(self, x, mask=None):
        p = self.patchify(x); h = self.proj(p)
        if mask is not None: h = torch.where(mask.unsqueeze(-1), self.mask_tok.expand_as(h), h)
        return self.norm(self.enc(h + self.pos)), p
    @torch.no_grad()
    def taps(self, x):
        # representations at four depths, for the depth analysis
        h = self.proj(self.patchify(x)) + self.pos
        out = {"proj": h.mean(1)}
        z = h
        for i, layer in enumerate(self.enc.layers, 1):
            z = layer(z)
            if i < len(self.enc.layers): out[f"layer{i}"] = z.mean(1)
        out["layer3_norm"] = self.norm(z).mean(1)
        return out

def pretrain(Xtr, epochs=EPOCHS, seed=42, bs=BATCH, lr=3e-4, verbose=False,
             n_tok=N_TOK, patch=PATCH):
    seed_all(seed)
    m = EMGEncoder(n_tok=n_tok, patch=patch).to(DEV)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.01)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    X = torch.from_numpy(Xtr).to(DEV); n = len(X)
    nm = max(1, int(round(MASK_RATIO * n_tok)))
    for ep in range(epochs):
        m.train(); perm = torch.randperm(n, device=DEV); tot = cnt = 0.0
        for i in range(0, n, bs):
            xb = X[perm[i:i+bs]]; B = len(xb)
            r = torch.rand(B, n_tok, device=DEV)
            mask = torch.zeros(B, n_tok, dtype=torch.bool, device=DEV)
            mask.scatter_(1, r.argsort(1)[:, :nm], True)
            h, tgt = m(xb, mask); loss = ((m.head(h) - tgt) ** 2)[mask].mean()
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
            tot += loss.item()*B; cnt += B
        sch.step()
        if verbose and ep % 5 == 0: print(f"  ep {ep:3d} masked-MSE {tot/cnt:.4f}")
    return m, tot/cnt

@torch.no_grad()
def embed(m, X, bs=256, tap=None):
    m.eval(); out = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i+bs]).to(DEV)
        out.append((m.taps(xb)[tap] if tap else m(xb)[0].mean(1)).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

#print("encoder parameters, 5 s config:", sum(p.numel() for p in EMGEncoder().parameters()))
#print("encoder parameters, 200 ms config:", sum(p.numel() for p in EMGEncoder(n_tok=N_TOK_S, patch=PATCH_S).parameters()))


# ===== CELL 10 =====
import torch.nn.functional as Fn
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from scipy.stats import beta as Beta, ttest_rel, spearmanr, t as tdist

class Baseline(nn.Module):
    def __init__(s, d, k):
        super().__init__(); s.lin = nn.Linear(d, k)
        s.logT = nn.Parameter(torch.zeros(1), requires_grad=False)
    def raw(s, z): return s.lin(z)
    def forward(s, z): return s.raw(z) / s.logT.exp()

class EnvHead(nn.Module):
    def __init__(s, d, k, delta=DELTA):
        super().__init__(); s.delta = delta
        s.net = nn.Sequential(nn.Linear(d,256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256,k))
        nn.init.zeros_(s.net[3].weight); nn.init.zeros_(s.net[3].bias)
    def corr(s, z): return s.net(z).clamp(-s.delta, s.delta)

class Envelope(nn.Module):
    def __init__(s, base, head): super().__init__(); s.base, s.head = base, head
    def forward(s, z):
        with torch.no_grad(): b = s.base(z)
        return b + s.head.corr(z), b

def fit_envelope(Zf, yf, Zc, yc, k, seed=42, delta=DELTA):
    torch.manual_seed(seed)
    base = Baseline(Zf.shape[1], k)
    Z, Y = torch.from_numpy(Zf), torch.from_numpy(yf)
    o = torch.optim.AdamW(base.lin.parameters(), lr=1e-2, weight_decay=1e-3)
    for _ in range(300):
        o.zero_grad(); Fn.cross_entropy(base.raw(Z), Y).backward(); o.step()
    with torch.no_grad(): lc = base.raw(torch.from_numpy(Zc))
    lt = torch.zeros(1, requires_grad=True)
    o2 = torch.optim.LBFGS([lt], lr=0.1, max_iter=100)
    def cl():
        o2.zero_grad(); l = Fn.cross_entropy(lc/lt.exp(), torch.from_numpy(yc)); l.backward(); return l
    o2.step(cl)
    with torch.no_grad(): base.logT.copy_(lt.detach())
    head = EnvHead(Zf.shape[1], k, delta); env = Envelope(base, head)
    o3 = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    for _ in range(400):
        o3.zero_grad(); f, _ = env(Z); Fn.cross_entropy(f, Y).backward(); o3.step()
    return base, env, float(lt.exp())

class Gate:
    def fit(s, Z, y, k, ridge=COV_RIDGE, pct=OOD_PCT):
        s.mu = np.stack([Z[y==c].mean(0) for c in range(k)])
        C = np.zeros((Z.shape[1],)*2)
        for c in range(k):
            d = Z[y==c] - s.mu[c]; C += d.T @ d
        C /= len(Z); C += ridge*np.trace(C)/C.shape[0]*np.eye(C.shape[0])
        s.P = np.linalg.inv(C); s.thr = np.percentile(s.score(Z), pct); return s
    def score(s, Z): return np.stack([((Z-m)@s.P*(Z-m)).sum(1) for m in s.mu]).min(0)
    def refuse(s, Z): return s.score(Z) > s.thr

def stats_all(env, gate, Z, base):
    with torch.no_grad(): f, _ = env(torch.from_numpy(Z))
    lg = f.numpy(); p = torch.softmax(f,1).numpy()
    part = np.partition(p, -2, 1); T = float(base.logT.exp())
    torch.manual_seed(0); env.head.train(); P = []
    with torch.no_grad():
        b = base(torch.from_numpy(Z))
        for _ in range(30): P.append(torch.softmax(b + env.head.corr(torch.from_numpy(Z)),1).numpy())
    env.head.eval(); P = np.stack(P); mp = P.mean(0).argmax(1)
    return p, dict(density=gate.score(Z), confidence=-p.max(1),
                   entropy=-(p*np.log(p+1e-12)).sum(1),
                   margin=-(part[:,-1]-part[:,-2]),
                   energy=-(np.log(np.exp(lg/T).sum(1))*T),
                   ensemble=P.var(0)[np.arange(len(mp)), mp])

def aurc(sc, correct, n=60):
    o = np.argsort(sc); c = correct[o]; cov = np.linspace(0.05,1,n)
    risk = np.array([1-c[:max(1,int(round(f*len(c))))].mean() for f in cov])
    return float(np.trapezoid(risk, cov)/(cov[-1]-cov[0]))

def ece(p, y, bins=15):
    conf, pred = p.max(1), p.argmax(1); acc = (pred==y).astype(float)
    e, ed = 0.0, np.linspace(0,1,bins+1)
    for i in range(bins):
        m = (conf>ed[i]) & (conf<=ed[i+1])
        if m.sum(): e += m.mean()*abs(acc[m].mean()-conf[m].mean())
    return float(e)
#print("envelope and statistics defined")


# ===== CELL 12 =====
RNG = np.random.RandomState(0)
def f_rotation(X): return np.roll(X, 3, axis=2)
def f_dropout(X):
    Xc = X.copy()
    for i in range(len(Xc)): Xc[i][:, RNG.choice(8,3,replace=False)] = 0.0
    return Xc
def f_gain(X):    return X * RNG.uniform(0.3, 2.5, (len(X),1,8)).astype(np.float32)
def f_artifact(X):
    Xc = X.copy()
    for i in range(len(Xc)):
        pos = RNG.choice(X.shape[1], 6, replace=False)
        Xc[i][pos] += RNG.choice([-1,1],(6,8)).astype(np.float32)*8.0
    return Xc
def f_noise(X):   return X + RNG.randn(*X.shape).astype(np.float32)*2.0
FAULTS = [("broadband noise", f_noise), ("gain shift", f_gain),
          ("channel dropout", f_dropout), ("channel rotation", f_rotation),
          ("motion artefact", f_artifact)]
#print("faults:", [f[0] for f in FAULTS])


A=np.load(os.path.join(DATA, "d1.npz")); X1,y1,s1,ss1=A["X"],A["y"],A["subj"],A["sess"]
B=np.load(os.path.join(DATA, "d2.npz")); X2,y2,s2=B["X"],B["y"],B["subj"]
IDX = dict(
    source_train = np.where(np.isin(s1, range(1,21)) & np.isin(ss1, [1,2,3,4]))[0],
    source_test  = np.where(np.isin(s1, range(1,21)) & (ss1 == 5))[0],
    target       = np.where(np.isin(s1, range(21,31)))[0])
ST=IDX["source_train"]
MU, SD = norm_fit(X1[ST])
norm = lambda X: ((X - MU) / SD).astype(np.float32)
keep = np.isin(y1, list(D1_MAP)); ym = np.array([D1_MAP.get(v,-1) for v in y1])
sub3 = lambda name: IDX[name][keep[IDX[name]]]


In [ ]:
%%writefile /content/emg_code/lib2.py
"""Shared helpers for the revision experiments: deployment conditions, classical
baseline, probe-only statistics, detection metrics."""
import numpy as np, torch
from scipy.signal import butter, sosfiltfilt, resample_poly
from scipy.ndimage import uniform_filter1d
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from core import stats_all, Gate

FS = 200

# ---------------------------------------------------------------- conditions
# Each deployment condition maps a RAW window batch (B,1000,8) to the input the
# model receives, given the training conditioning `conv` and the training
# normalization (mu, sd). The model always expects norm(conv(x)).
def _bp(X, lo=20.0, hi=90.0):
    sos = butter(4, [lo, hi], btype="band", fs=FS, output="sos")
    return sosfiltfilt(sos, X, axis=1).astype(np.float32)

def _up2(X):
    Y = resample_poly(X, 2, 1, axis=1).astype(np.float32)
    n = X.shape[1]; s = (Y.shape[1] - n) // 2
    return Y[:, s:s + n]

def make_conditions(conv, mu, sd, signed_raw_available, rect_train):
    """Return dict name -> f(Xraw) giving the model input. `conv` is the training
    conditioning applied to raw data (abs for rectified, identity for signed)."""
    N = lambda X: ((X - mu) / sd).astype(np.float32)
    C = {}
    C["matched"] = lambda X: N(conv(X))
    for g in [0.1, 0.25, 0.5, 2.0, 4.0, 10.0]:
        C[f"gain x{g:g}"] = (lambda g: lambda X: N(conv(X) * g))(g)
    C["envelope 50 ms"] = lambda X: N(uniform_filter1d(conv(X), 10, axis=1))
    C["envelope 150 ms"] = lambda X: N(uniform_filter1d(conv(X), 30, axis=1))
    def perwin(X):
        Y = conv(X); m = Y.mean(1, keepdims=True); s = Y.std(1, keepdims=True) + 1e-6
        return ((Y - m) / s).astype(np.float32)
    C["per-window z-score"] = perwin
    if rect_train:
        C["400 Hz stream"] = lambda X: N(np.maximum(_up2(conv(X)), 0))
    else:
        C["400 Hz stream"] = lambda X: N(_up2(conv(X)))
    if signed_raw_available:
        C["band-pass 20-90 Hz"] = lambda X: N(conv(_bp(X)))
        if rect_train:
            C["signed (no rectification)"] = lambda X: N(X)
        else:
            C["rectified"] = lambda X: N(np.abs(X))
    return C

# ---------------------------------------------------------------- probe-only
def probe_stats(base, Z):
    with torch.no_grad():
        lg = base(torch.from_numpy(Z)).numpy()
    p = np.exp(lg - lg.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    return p, -p.max(1)

def all_stats(env, gate, Z, base):
    p, sc = stats_all(env, gate, Z, base)
    pp, cp = probe_stats(base, Z)
    sc["confidence (probe only)"] = cp
    return p, pp, sc

def auroc(s_in, s_out):
    lab = np.r_[np.zeros(len(s_in)), np.ones(len(s_out))]
    return float(roc_auc_score(lab, np.r_[s_in, s_out]))

def tpr_at(s_in, s_out, fr=0.05):
    thr = np.percentile(s_in, 100 * (1 - fr))
    return float((s_out > thr).mean())

def contract_score(Xin, mu, sd, lo, hi):
    """Fraction of samples of the model input, mapped back to raw units, that fall
    outside the range seen in (conditioned) training data."""
    R = Xin * sd + mu
    return ((R < lo - 1e-3) | (R > hi + 1e-3)).mean(axis=(1, 2))

# ---------------------------------------------------------------- classical
def td_features(X, thr=1.0):
    """Hudgins time-domain set per channel on the model input (raw units)."""
    mav = np.abs(X).mean(1)
    wl = np.abs(np.diff(X, axis=1)).sum(1) / X.shape[1]
    a, b = X[:, :-1], X[:, 1:]
    zc = (((a * b) < 0) & (np.abs(a - b) >= thr)).mean(1)
    d1, d2 = X[:, 1:-1] - X[:, :-2], X[:, 1:-1] - X[:, 2:]
    ssc = (((d1 * d2) > 0) & ((np.abs(d1) >= thr) | (np.abs(d2) >= thr))).mean(1)
    return np.concatenate([np.log(mav + 1e-3), np.log(wl + 1e-3), zc, ssc], 1).astype(np.float64)

class LDABaseline:
    def fit(self, F, y, k):
        self.m, self.s = F.mean(0), F.std(0)
        self.s[self.s < 1e-8] = 1.0
        Fz = (F - self.m) / self.s
        self.lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto").fit(Fz, y)
        self.gate = Gate().fit(Fz, y, k)
        return self
    def run(self, F):
        Fz = (F - self.m) / self.s
        p = self.lda.predict_proba(Fz)
        return p, dict(confidence=-p.max(1), density=self.gate.score(Fz))


In [ ]:
%%writefile /content/emg_code/lib3.py
"""Probe-only classifier and statistics for the revised paper (no residual head)."""
import numpy as np, torch
import torch.nn.functional as Fn
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from core import Baseline, Gate, ece, aurc

def fit_probe(Zf, yf, Zc, yc, k, seed=42):
    """Multinomial linear probe (AdamW, 300 full-batch steps) followed by
    temperature scaling on the calibration participants (L-BFGS)."""
    torch.manual_seed(seed)
    base = Baseline(Zf.shape[1], k)
    Z, Y = torch.from_numpy(Zf), torch.from_numpy(yf)
    o = torch.optim.AdamW(base.lin.parameters(), lr=1e-2, weight_decay=1e-3)
    for _ in range(300):
        o.zero_grad(); Fn.cross_entropy(base.raw(Z), Y).backward(); o.step()
    with torch.no_grad(): lc = base.raw(torch.from_numpy(Zc))
    lt = torch.zeros(1, requires_grad=True)
    o2 = torch.optim.LBFGS([lt], lr=0.1, max_iter=100)
    def cl():
        o2.zero_grad(); l = Fn.cross_entropy(lc / lt.exp(), torch.from_numpy(yc)); l.backward(); return l
    o2.step(cl)
    with torch.no_grad(): base.logT.copy_(lt.detach())
    return base

STATS = ["density", "confidence", "entropy", "margin", "energy"]

def pstats(base, gate, Z):
    with torch.no_grad():
        raw = base.raw(torch.from_numpy(Z)).numpy()
        lg = base(torch.from_numpy(Z)).numpy()          # calibrated logits
    p = np.exp(lg - lg.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
    part = np.partition(p, -2, 1)
    m = raw.max(1, keepdims=True)
    energy = -(m[:, 0] + np.log(np.exp(raw - m).sum(1)))   # negative log-sum-exp of probe logits
    return p, dict(density=gate.score(Z), confidence=-p.max(1),
                   entropy=-(p * np.log(p + 1e-12)).sum(1),
                   margin=-(part[:, -1] - part[:, -2]), energy=energy)

def auroc(s_in, s_out):
    lab = np.r_[np.zeros(len(s_in)), np.ones(len(s_out))]
    return float(roc_auc_score(lab, np.r_[s_in, s_out]))

def tpr_at(s_in, s_out, fr=0.05):
    return float((s_out > np.percentile(s_in, 100 * (1 - fr))).mean())

def macro_auc(p, y, k):
    pres = [c for c in range(k) if (y == c).any() and (y != c).any()]
    return float(np.mean([roc_auc_score((y == c).astype(int), p[:, c]) for c in pres]))

def summary(p, y, k, gate, Z):
    pr = p.argmax(1); ref = gate.refuse(Z)
    out = dict(acc=float((pr == y).mean()), bal=float(balanced_accuracy_score(y, pr)),
               mauc=macro_auc(p, y, k), ece=ece(p, y), conf=float(p.max(1).mean()),
               refusal=float(ref.mean()),
               recall=[float((pr[y == c] == c).mean()) if (y == c).any() else np.nan for c in range(k)],
               pred_hist=np.bincount(pr, minlength=k).tolist())
    if ref.sum() > 5 and (~ref).sum() > 5:
        out["bal_accepted"] = float(balanced_accuracy_score(y[~ref], pr[~ref]))
        out["bal_refused"] = float(balanced_accuracy_score(y[ref], pr[ref]))
        out["err_accepted"] = float((pr[~ref] != y[~ref]).mean())
        out["err_refused"] = float((pr[ref] != y[ref]).mean())
    return out


## 5. Experiments
Each script writes one result file per seed or split and skips files that already exist.

In [ ]:
%%writefile /content/emg_code/v3_a.py
"""Revised paper, Dataset A main split, ten seeds, calibrated linear probe."""
import sys, os
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
RES = os.environ.get("EMG_RES", os.path.join(HERE, "results"))
import time, pickle
from core import *
import core
from lib2 import make_conditions, contract_score, td_features, LDABaseline
from lib3 import *

OUT = os.environ.get("OUT", os.path.join(RES, "v3_a")); os.makedirs(OUT, exist_ok=True)
SEEDS = [int(s) for s in os.environ.get("SEEDS", "42,43,44,45,46,47,48,49,50,51").split(",")]
is_cal = np.isin(s1[ST], CALIB_SUBJECTS); FIT, CAL = ST[~is_cal], ST[is_cal]
TE, TG = IDX["source_test"], IDX["target"]
LO, HI = float(X1[ST].min()), float(X1[ST].max())
CONDS = make_conditions(lambda X: X, MU, SD, False, True); CONDS.pop("matched")
FAULT_FN = dict(core.FAULTS)
POOL = list(FAULT_FN) + ["gain x4", "gain x0.25", "envelope 150 ms", "per-window z-score", "400 Hz stream"]
Xn1 = norm(X1)

def apply_pool(Xraw, names):
    X = Xraw.copy(); post = False
    for nm in names:
        if nm in FAULT_FN: X = FAULT_FN[nm](X)
        elif nm == "per-window z-score": post = True
        else: X = CONDS[nm](X) * SD + MU
    return CONDS["per-window z-score"](X) if post else norm(X)

def shifted(nm, f):
    Xin = norm(f(X1[TE])) if nm in FAULT_FN else f(X1[TE])
    return Xin

def run(seed):
    fn = f"{OUT}/a_{seed}.pkl"
    if os.path.exists(fn): return
    t0 = time.time(); core.RNG = np.random.RandomState(seed)
    enc, rec = pretrain(Xn1[ST], seed=seed)
    torch.save(enc.state_dict(), f"{OUT}/enc_{seed}.pt")
    Zr = embed(enc, Xn1); m0, s0 = Zr[ST].mean(0), Zr[ST].std(0) + 1e-6
    Zs = lambda Xin: ((embed(enc, Xin) - m0) / s0).astype(np.float32)
    Z = ((Zr - m0) / s0).astype(np.float32)
    base = fit_probe(Z[FIT], y1[FIT], Z[CAL], y1[CAL], K6, seed)
    gate = Gate().fit(Z[FIT], y1[FIT], K6)
    R = dict(seed=seed, recon=rec, T=float(base.logT.exp()))

    # six-class ladder
    for nm, v in [("source_test", TE), ("target", TG)]:
        p, sc = pstats(base, gate, Z[v]); R[nm] = summary(p, y1[v], K6, gate, Z[v])
        if nm == "target":
            corr = (p.argmax(1) == y1[v]).astype(float)
            R[nm]["aurc"] = {k: aurc(sc[k], corr) for k in STATS}
            R[nm]["scores"] = {k: sc[k].astype(np.float32) for k in STATS}
            R[nm]["correct"] = corr.astype(np.float32)
            R["per_participant"] = {int(j): dict(acc=float(corr[s1[v] == j].mean()),
                                                 bal=float(balanced_accuracy_score(y1[v][s1[v] == j], p.argmax(1)[s1[v] == j])),
                                                 conf=float(p.max(1)[s1[v] == j].mean())) for j in range(21, 31)}
        else:
            sc_in = sc; sc_in["range check"] = contract_score(Xn1[TE], MU, SD, LO, HI)

    # untrained encoder control
    seed_all(seed); encr = EMGEncoder().to(DEV).eval()
    Zu = embed(encr, Xn1); Zu = ((Zu - Zu[ST].mean(0)) / (Zu[ST].std(0) + 1e-6)).astype(np.float32)
    bu = fit_probe(Zu[FIT], y1[FIT], Zu[CAL], y1[CAL], K6, seed); gu = Gate().fit(Zu[FIT], y1[FIT], K6)
    pu, su = pstats(bu, gu, Zu[TG]); cu = (pu.argmax(1) == y1[TG]).astype(float)
    R["untrained"] = dict(acc=float(cu.mean()), bal=float(balanced_accuracy_score(y1[TG], pu.argmax(1))),
                          aurc={k: aurc(su[k], cu) for k in STATS})

    # faults and mismatch family, six-class
    R["shift"] = {}
    for nm, f in list(FAULT_FN.items()) + list(CONDS.items()):
        Xin = shifted(nm, f); Zo = Zs(Xin)
        po, so = pstats(base, gate, Zo); so["range check"] = contract_score(Xin, MU, SD, LO, HI)
        R["shift"][nm] = dict(**summary(po, y1[TE], K6, gate, Zo),
                              auroc={k: auroc(sc_in[k], so[k]) for k in so},
                              tpr5={k: tpr_at(sc_in[k], so[k]) for k in STATS})

    # three-class ladder with Dataset B and the factorial
    st3 = sub3("source_train"); ic = np.isin(s1[st3], CALIB_SUBJECTS)
    b3 = fit_probe(Z[st3[~ic]], ym[st3[~ic]], Z[st3[ic]], ym[st3[ic]], KC, seed)
    g3 = Gate().fit(Z[st3[~ic]], ym[st3[~ic]], KC)
    muBr, sdBr = norm_fit(np.abs(X2)); muBs, sdBs = norm_fit(X2)
    L = {}; ref = None
    for nm, Xin, yq in [("source_test", Xn1[sub3("source_test")], ym[sub3("source_test")]),
                        ("cross-participant", Xn1[sub3("target")], ym[sub3("target")]),
                        ("cross-dataset", norm(np.abs(X2)), y2), ("mismatch", norm(X2), y2),
                        ("signed + own norm", ((X2 - muBs) / sdBs).astype(np.float32), y2),
                        ("rectified + own norm", ((np.abs(X2) - muBr) / sdBr).astype(np.float32), y2)]:
        Zq = Zs(Xin); p3, s3 = pstats(b3, g3, Zq); s3["range check"] = contract_score(Xin, MU, SD, LO, HI)
        r = summary(p3, yq, KC, g3, Zq)
        if ref is None: ref = s3
        else:
            r["auroc"] = {k: auroc(ref[k], s3[k]) for k in s3}
            r["tpr5"] = {k: tpr_at(ref[k], s3[k]) for k in STATS}
        L[nm] = r
    R["ladder3"] = L

    # cascade on cross-participant windows
    rng = np.random.RandomState(seed + 7); n = len(TG)
    faulty = rng.rand(n) < 0.5
    combos = [list(rng.choice(POOL, rng.randint(1, 3), replace=False)) if faulty[i] else [] for i in range(n)]
    Xin = np.stack([apply_pool(X1[TG][i:i+1], combos[i])[0] for i in range(n)])
    Zm = Zs(Xin); pm, sm = pstats(base, gate, Zm)
    chk = contract_score(Xin, MU, SD, LO, HI) > 0
    pc, _ = pstats(base, gate, Z[CAL]); cthr = np.percentile(pc.max(1), 10)
    conf_ok = pm.max(1) >= cthr; dens_ok = ~gate.refuse(Zm)
    pol = {"none": np.ones(n, bool), "confidence": conf_ok, "density": dens_ok, "range check": ~chk,
           "range + density": ~chk & dens_ok, "range + density + confidence": ~chk & dens_ok & conf_ok}
    yt = y1[TG]; pred = pm.argmax(1); C = {}
    for k, a in pol.items():
        C[k] = dict(cov_clean=float(a[~faulty].mean()), cov_faulty=float(a[faulty].mean()),
                    bal_accepted=float(balanced_accuracy_score(yt[a], pred[a])),
                    err_accepted=float((pred[a] != yt[a]).mean()),
                    harmful=float((a & (pred != yt) & faulty).mean()),
                    harmful_share=float((a & (pred != yt) & faulty).sum() / max(1, a.sum())))
    R["cascade"] = C; R["cascade_meta"] = dict(n=n, n_faulty=int(faulty.sum()), conf_thr=float(cthr),
                                               err_faulty_none=float((pred[faulty] != yt[faulty]).mean()),
                                               err_clean_none=float((pred[~faulty] != yt[~faulty]).mean()))
    if seed == SEEDS[0]:
        x = torch.from_numpy(Xn1[:1]).to(DEV); enc.eval()
        with torch.no_grad():
            for _ in range(5): enc(x)
            t1 = time.perf_counter()
            for _ in range(300): enc(x)
            lat = (time.perf_counter() - t1) / 300
        zz = Z[:1]; t1 = time.perf_counter()
        for _ in range(3000): gate.score(zz)
        R["latency"] = dict(encoder_ms=lat * 1e3, gate_ms=(time.perf_counter() - t1) / 3 ,
                            params=int(sum(q.numel() for q in enc.parameters())))
    pickle.dump(R, open(fn, "wb"))
    mm = R["ladder3"]["mismatch"]
    print(f"[A {seed}] {time.time()-t0:.0f}s tgt acc {R['target']['acc']:.3f} | mismatch bal {mm['bal']:.3f} conf {mm['conf']:.4f} "
          f"AUROC conf {mm['auroc']['confidence']:.3f} dens {mm['auroc']['density']:.3f} | cascade harmful "
          f"{C['none']['harmful']:.3f}->{C['range + density + confidence']['harmful']:.3f}", flush=True)

for s in SEEDS: run(s)
print("A DONE", flush=True)


In [ ]:
# Dataset A, main split: ladder, faults, mismatch family, error ranking, mixed faults (Tables III, IV, VI, VIII, X; Figs. 2, 4 to 7)
run("v3_a.py", SEEDS=os.environ["SEEDS_A"])


In [ ]:
%%writefile /content/emg_code/v3_b.py
"""Within-Dataset-B conditioning, ten participant splits, calibrated probe."""
import sys, os
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
RES = os.environ.get("EMG_RES", os.path.join(HERE, "results"))
import time, pickle
from core import *
from lib2 import make_conditions, contract_score
from lib3 import *
OUT = os.environ.get("OUT", os.path.join(RES, "v3_b")); os.makedirs(OUT, exist_ok=True)
SEEDS = [int(s) for s in os.environ.get("SEEDS", "0,1,2,3,4,5,6,7,8,9").split(",")]
pids = np.unique(s2)
def run(seed, tc):
    fn = f"{OUT}/{tc}_{seed}.pkl"
    if os.path.exists(fn): return
    t0 = time.time()
    perm = np.random.RandomState(100 + seed).permutation(pids)
    tr_p, te_p = perm[:26], perm[26:]; cal_p = tr_p[-4:]
    itr = np.where(np.isin(s2, tr_p))[0]; ite = np.where(np.isin(s2, te_p))[0]
    cal = np.isin(s2[itr], cal_p); fit, cl = itr[~cal], itr[cal]
    if tc == "mixed":
        Xtr_c = np.concatenate([np.abs(X2[itr]), X2[itr]]); conv = np.abs; ep = 13
    else:
        conv = np.abs if tc == "rectified" else (lambda X: X); Xtr_c = conv(X2[itr]); ep = EPOCHS
    mu, sd = norm_fit(Xtr_c); lo, hi = float(Xtr_c.min()), float(Xtr_c.max())
    N = lambda X: ((X - mu) / sd).astype(np.float32)
    enc, rec = pretrain(N(Xtr_c), epochs=ep, seed=seed)
    Ztr = embed(enc, N(Xtr_c)); m0, s0 = Ztr.mean(0), Ztr.std(0) + 1e-6
    Zs = lambda Xin: ((embed(enc, Xin) - m0) / s0).astype(np.float32)
    if tc == "mixed":
        Zf = np.concatenate([Zs(N(np.abs(X2[fit]))), Zs(N(X2[fit]))]); yf = np.r_[y2[fit], y2[fit]]
        Zc = np.concatenate([Zs(N(np.abs(X2[cl]))), Zs(N(X2[cl]))]); yc = np.r_[y2[cl], y2[cl]]
    else:
        Zf, yf, Zc, yc = Zs(N(conv(X2[fit]))), y2[fit], Zs(N(conv(X2[cl]))), y2[cl]
    base = fit_probe(Zf, yf, Zc, yc, KC, seed); gate = Gate().fit(Zf, yf, KC)
    conds = make_conditions(conv, mu, sd, True, tc != "signed")
    if tc == "mixed": conds["signed (no rectification)"] = lambda X: N(X)
    yt = y2[ite]; R = dict(seed=seed, train=tc, test=te_p.tolist(), n_test=len(ite),
                           n_fit=len(fit), n_cal=len(cl), cond={}, counts_test=np.bincount(yt, minlength=3).tolist())
    ref = None
    for nm, f in conds.items():
        Xin = f(X2[ite]); Z = Zs(Xin); p, sc = pstats(base, gate, Z)
        sc["range check"] = contract_score(Xin, mu, sd, lo, hi)
        r = summary(p, yt, KC, gate, Z)
        if nm == "matched": ref = sc
        else:
            r["auroc"] = {k: auroc(ref[k], sc[k]) for k in sc}
            r["tpr5"] = {k: tpr_at(ref[k], sc[k]) for k in STATS}
        R["cond"][nm] = r
    pickle.dump(R, open(fn, "wb"))
    key = {"rectified": "signed (no rectification)", "signed": "rectified", "mixed": "gain x0.1"}[tc]
    c = R["cond"][key]
    print(f"[B {tc} {seed}] {time.time()-t0:.0f}s matched {R['cond']['matched']['bal']:.3f} | {key} bal {c['bal']:.3f} "
          f"conf {c['auroc']['confidence']:.3f} dens {c['auroc']['density']:.3f}", flush=True)
for s in SEEDS:
    for tc in ["rectified", "signed", "mixed"]: run(s, tc)
print("B DONE", flush=True)


In [ ]:
# Within Dataset B: rectified, signed and both-convention training, ten participant splits (Tables V, VI)
run("v3_b.py", SEEDS=os.environ["SEEDS_B"])


In [ ]:
%%writefile /content/emg_code/v3_splits.py
"""Ten random 20/10 participant splits: depth and conditioning mismatch, calibrated probe."""
import sys, os
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
RES = os.environ.get("EMG_RES", os.path.join(HERE, "results"))
import time, pickle
from core import *
import core
from lib2 import contract_score
from lib3 import *
OUT = os.environ.get("OUT", os.path.join(RES, "v3_s")); os.makedirs(OUT, exist_ok=True)
FAULT_FN = dict(core.FAULTS)
for r in range(int(os.environ.get("NS", 10))):
    fn = f"{OUT}/s_{r}.pkl"
    if os.path.exists(fn): continue
    t0 = time.time()
    perm = np.random.RandomState(1000 + r).permutation(np.arange(1, 31))
    tr_p, te_p = perm[:20], perm[20:]; cal_p = tr_p[-4:]
    itr = np.where(np.isin(s1, tr_p) & np.isin(ss1, [1, 2, 3, 4]))[0]
    ite = np.where(np.isin(s1, te_p))[0]; iref = np.where(np.isin(s1, tr_p) & (ss1 == 5))[0]
    mu_, sd_ = norm_fit(X1[itr]); Nf = lambda X: ((X - mu_) / sd_).astype(np.float32); Xr = Nf(X1)
    enc, _ = pretrain(Xr[itr], seed=42 + r)
    isc = np.isin(s1[itr], cal_p); fi, ci = itr[~isc], itr[isc]
    R = dict(split=r, test=te_p.tolist(), cal=cal_p.tolist(), depth={})
    core.RNG = np.random.RandomState(r)
    faulted = {nm: Nf(f(X1[iref])) for nm, f in FAULT_FN.items()}
    for tap in ["proj", "layer1", "layer2", "layer3_norm"]:
        Zt = embed(enc, Xr, tap=tap); mt, st_ = Zt[itr].mean(0), Zt[itr].std(0) + 1e-6
        Zt = ((Zt - mt) / st_).astype(np.float32)
        bt = fit_probe(Zt[fi], y1[fi], Zt[ci], y1[ci], K6, 42 + r); gt = Gate().fit(Zt[fi], y1[fi], K6)
        pin, sin = pstats(bt, gt, Zt[iref]); oods = {}
        for nm, Xf in faulted.items():
            Zo = ((embed(enc, Xf, tap=tap) - mt) / st_).astype(np.float32)
            oods[nm] = auroc(sin["density"], gt.score(Zo))
        pt, stt = pstats(bt, gt, Zt[ite]); pr = pt.argmax(1); corr = (pr == y1[ite]).astype(float)
        R["depth"][tap] = dict(acc=float(corr.mean()), bal=float(balanced_accuracy_score(y1[ite], pr)),
                               ood=oods, mean_ood=float(np.mean(list(oods.values()))),
                               aurc_conf=aurc(stt["confidence"], corr))
    d = R["depth"]["layer3_norm"]; R["acc"] = d["acc"]; R["bal"] = d["bal"]
    Zr = embed(enc, Xr); m0, s0 = Zr[itr].mean(0), Zr[itr].std(0) + 1e-6
    Zs = lambda Xin: ((embed(enc, Xin) - m0) / s0).astype(np.float32); Z = ((Zr - m0) / s0).astype(np.float32)
    k3 = keep[itr]; f3, c3 = itr[k3 & ~isc], itr[k3 & isc]
    b3 = fit_probe(Z[f3], ym[f3], Z[c3], ym[c3], KC, 42 + r); g3 = Gate().fit(Z[f3], ym[f3], KC)
    r3 = iref[keep[iref]]; _, sref = pstats(b3, g3, Z[r3])
    for nm, Xq in [("cross-dataset", Nf(np.abs(X2))), ("mismatch", Nf(X2))]:
        Zq = Zs(Xq); p3, s3 = pstats(b3, g3, Zq)
        R[nm] = dict(bal=float(balanced_accuracy_score(y2, p3.argmax(1))), conf=float(p3.max(1).mean()),
                     auroc={k: auroc(sref[k], s3[k]) for k in s3})
    pickle.dump(R, open(fn, "wb"))
    print(f"[split {r}] {time.time()-t0:.0f}s ood " + " ".join(f"{k}:{v['mean_ood']:.3f}" for k, v in R["depth"].items())
          + f" | mismatch bal {R['mismatch']['bal']:.3f} conf {R['mismatch']['auroc']['confidence']:.3f}", flush=True)
print("S DONE", flush=True)


In [ ]:
# Ten random participant splits of Dataset A: tap depth and conditioning mismatch (Table IX, Figs. 3, 9, 10)
run("v3_splits.py")


In [ ]:
%%writefile /content/emg_code/v3_nov.py
"""Held-out gestures excluded from encoder pretraining: four pairs x five seeds, probe."""
import sys, os
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
RES = os.environ.get("EMG_RES", os.path.join(HERE, "results"))
import time, pickle
from core import *
from lib3 import *
OUT = os.environ.get("OUT", os.path.join(RES, "v3_n")); os.makedirs(OUT, exist_ok=True)
PAIRS = [(4, 5), (1, 2), (3, 4), (2, 5)][:int(os.environ.get("NPAIRS", 4))]
for seed in [int(s) for s in os.environ.get("SEEDS", "42,43,44,45,46").split(",")]:
    for hp in PAIRS:
        fn = f"{OUT}/n_{seed}_{hp[0]}{hp[1]}.pkl"
        if os.path.exists(fn): continue
        t0 = time.time(); seen = [c for c in range(K6) if c not in hp]
        trs = ST[np.isin(y1[ST], seen)]
        mu_, sd_ = norm_fit(X1[trs]); Xr = ((X1 - mu_) / sd_).astype(np.float32)
        e_, _ = pretrain(Xr[trs], seed=seed)
        Zr = embed(e_, Xr); Z = ((Zr - Zr[trs].mean(0)) / (Zr[trs].std(0) + 1e-6)).astype(np.float32)
        rm = {c: i for i, c in enumerate(seen)}; cal = np.isin(s1[trs], CALIB_SUBJECTS); fi, ci = trs[~cal], trs[cal]
        b_ = fit_probe(Z[fi], np.array([rm[c] for c in y1[fi]]), Z[ci], np.array([rm[c] for c in y1[ci]]), 4, seed)
        g_ = Gate().fit(Z[fi], np.array([rm[c] for c in y1[fi]]), 4)
        te = IDX["source_test"]; inm = np.isin(y1[te], seen)
        _, si = pstats(b_, g_, Z[te[inm]]); _, so = pstats(b_, g_, Z[te[~inm]])
        R = dict(seed=seed, pair=hp, n_in=int(inm.sum()), n_out=int((~inm).sum()),
                 auroc={k: auroc(si[k], so[k]) for k in STATS})
        pickle.dump(R, open(fn, "wb"))
        print(f"[nov {seed} {hp}] {time.time()-t0:.0f}s dens {R['auroc']['density']:.3f} conf {R['auroc']['confidence']:.3f}", flush=True)
print("N DONE", flush=True)


In [ ]:
# Held-out gestures excluded from pretraining: four gesture pairs x five seeds (Table IV)
run("v3_nov.py", SEEDS=os.environ["SEEDS_N"])


In [ ]:
%%writefile /content/emg_code/v3_200.py
"""200 ms decision windows, five seeds, calibrated probe."""
import sys, os
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
RES = os.environ.get("EMG_RES", os.path.join(HERE, "results"))
import time, pickle
from core import *
from lib3 import *
OUT = os.environ.get("OUT", os.path.join(RES, "v3_w")); os.makedirs(OUT, exist_ok=True)
WIN = N_TOK_S * PATCH_S
def rewindow(X, *vals):
    n = X.shape[1] // WIN
    return [X[:, :n * WIN].reshape(len(X) * n, WIN, N_CH)] + [np.repeat(v, n) for v in vals]
Xs1, ys1, sj1, se1 = rewindow(X1, y1, s1, ss1); Xs2, ys2 = rewindow(X2, y2)
keepS = np.isin(ys1, list(D1_MAP)); ymS = np.array([D1_MAP.get(v, -1) for v in ys1])
itr = np.where(np.isin(sj1, range(1, 21)) & np.isin(se1, [1, 2, 3, 4]))[0]
ite = np.where(np.isin(sj1, range(1, 21)) & (se1 == 5))[0]; itg = np.where(sj1 > 20)[0]
mu_, sd_ = norm_fit(Xs1[itr]); nz = lambda X: ((X - mu_) / sd_).astype(np.float32); XN = nz(Xs1)
for seed in [int(s) for s in os.environ.get("SEEDS", "42,43,44,45,46").split(",")]:
    fn = f"{OUT}/w_{seed}.pkl"
    if os.path.exists(fn): continue
    t0 = time.time()
    enc, _ = pretrain(XN[itr], seed=seed, n_tok=N_TOK_S, patch=PATCH_S)
    Zr = embed(enc, XN); a0, b0 = Zr[itr].mean(0), Zr[itr].std(0) + 1e-6
    Zn = lambda A_: ((A_ - a0) / b0).astype(np.float32); ZS = Zn(Zr)
    tr3 = itr[keepS[itr]]; cal = np.isin(sj1[tr3], CALIB_SUBJECTS)
    b = fit_probe(ZS[tr3[~cal]], ymS[tr3[~cal]], ZS[tr3[cal]], ymS[tr3[cal]], KC, seed)
    g = Gate().fit(ZS[tr3[~cal]], ymS[tr3[~cal]], KC)
    R = dict(seed=seed, n_windows=dict(A=len(Xs1), B=len(Xs2)), cond={}); ref = None
    for nm, Zq, yq in [("source_test", ZS[ite[keepS[ite]]], ymS[ite[keepS[ite]]]),
                       ("cross-participant", ZS[itg[keepS[itg]]], ymS[itg[keepS[itg]]]),
                       ("cross-dataset", Zn(embed(enc, nz(np.abs(Xs2)))), ys2),
                       ("mismatch", Zn(embed(enc, nz(Xs2))), ys2)]:
        p, sc = pstats(b, g, Zq); r = summary(p, yq, KC, g, Zq); r["n"] = len(yq)
        if ref is None: ref = sc
        else: r["auroc"] = {k: auroc(ref[k], sc[k]) for k in STATS}
        R["cond"][nm] = r
    pickle.dump(R, open(fn, "wb"))
    m = R["cond"]["mismatch"]
    print(f"[200ms {seed}] {time.time()-t0:.0f}s mismatch bal {m['bal']:.3f} conf {m['conf']:.4f} AUROC conf {m['auroc']['confidence']:.3f} dens {m['auroc']['density']:.3f}", flush=True)
print("W DONE", flush=True)


In [ ]:
# 200 ms decision windows, separate 8-token encoder (Fig. 8)
run("v3_200.py", SEEDS=os.environ["SEEDS_W"])


In [ ]:
%%writefile /content/emg_code/v3_lda.py
"""Classical baseline: Hudgins time-domain features with shrinkage LDA, on Dataset A
(source test, mismatch family, three-class ladder) and within Dataset B (ten splits).
Deterministic; no neural network involved."""
import sys, os
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
RES = os.environ.get("EMG_RES", os.path.join(HERE, "results"))
import pickle
from core import *
from lib2 import make_conditions, td_features, LDABaseline
from lib3 import auroc
from sklearn.metrics import balanced_accuracy_score as BA

is_cal = np.isin(s1[ST], CALIB_SUBJECTS); FIT = ST[~is_cal]; TE = IDX["source_test"]
CONDS = make_conditions(lambda X: X, MU, SD, False, True); CONDS.pop("matched")
R = {"A": {}, "B": {}}
lda = LDABaseline().fit(td_features(X1[FIT]), y1[FIT], K6)
p0, s0 = lda.run(td_features(X1[TE]))
R["A"]["source_test_bal6"] = float(BA(y1[TE], p0.argmax(1)))
R["A"]["shift"] = {}
for nm, f in CONDS.items():
    p, s = lda.run(td_features(f(X1[TE]) * SD + MU))
    R["A"]["shift"][nm] = dict(bal=float(BA(y1[TE], p.argmax(1))), auroc={k: auroc(s0[k], s[k]) for k in s})
st3 = sub3("source_train"); ic = np.isin(s1[st3], CALIB_SUBJECTS)
lda3 = LDABaseline().fit(td_features(X1[st3[~ic]]), ym[st3[~ic]], KC)
L = {}; ref = None
for nm, Xraw, yq in [("source_test", X1[sub3("source_test")], ym[sub3("source_test")]),
                     ("cross-participant", X1[sub3("target")], ym[sub3("target")]),
                     ("cross-dataset", np.abs(X2), y2), ("mismatch", X2, y2)]:
    p, s = lda3.run(td_features(Xraw)); r = dict(bal=float(BA(yq, p.argmax(1))))
    if ref is None: ref = s
    else: r["auroc"] = {k: auroc(ref[k], s[k]) for k in s}
    L[nm] = r
R["A"]["ladder3"] = L
pids = np.unique(s2)
for tc in ["rectified", "signed", "mixed"]:
    R["B"][tc] = []
    for seed in [int(x) for x in os.environ.get("SEEDS_B", "0,1,2,3,4,5,6,7,8,9").split(",")]:
        perm = np.random.RandomState(100 + seed).permutation(pids)
        tr_p, te_p = perm[:26], perm[26:]; cal_p = tr_p[-4:]
        itr = np.where(np.isin(s2, tr_p))[0]; ite = np.where(np.isin(s2, te_p))[0]
        fit = itr[~np.isin(s2[itr], cal_p)]
        if tc == "mixed":
            conv = np.abs; Xtr_c = np.concatenate([np.abs(X2[itr]), X2[itr]])
            F = np.concatenate([td_features(np.abs(X2[fit])), td_features(X2[fit])]); yf = np.r_[y2[fit], y2[fit]]
        else:
            conv = np.abs if tc == "rectified" else (lambda X: X); Xtr_c = conv(X2[itr])
            F = td_features(conv(X2[fit])); yf = y2[fit]
        mu, sd = norm_fit(Xtr_c)
        conds = make_conditions(conv, mu, sd, True, tc != "signed")
        if tc == "mixed": conds["signed (no rectification)"] = lambda X: ((X - mu) / sd).astype(np.float32)
        lb = LDABaseline().fit(F, yf, KC); yt = y2[ite]; out = {}; refs = None
        for nm, f in conds.items():
            p, s = lb.run(td_features(f(X2[ite]) * sd + mu)); r = dict(bal=float(BA(yt, p.argmax(1))))
            if nm == "matched": refs = s
            else: r["auroc"] = {k: auroc(refs[k], s[k]) for k in s}
            out[nm] = r
        R["B"][tc].append(out)
pickle.dump(R, open(os.path.join(RES, "v3_lda.pkl"), "wb"))
print("LDA source test (6 classes):", round(R["A"]["source_test_bal6"], 3),
      "| within B signed:", round(np.mean([o["signed (no rectification)"]["bal"] for o in R["B"]["rectified"]]), 3))


In [ ]:
# Classical time-domain features with LDA (Table VII)
run("v3_lda.py")


In [ ]:
%%writefile /content/emg_code/v3_within.py
import sys, os
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
RES = os.environ.get("EMG_RES", os.path.join(HERE, "results"))
import pickle
from core import *; from lib3 import fit_probe
res = {}
for seed in [int(x) for x in os.environ.get("SEEDS", "42,43,44").split(",")]:
    enc = EMGEncoder().to(DEV); enc.load_state_dict(torch.load(os.path.join(RES, "v3_a", f"enc_{seed}.pt"), map_location=DEV)); enc.eval()
    Xn1 = norm(X1); Zr = embed(enc, Xn1); Z = ((Zr - Zr[ST].mean(0)) / (Zr[ST].std(0) + 1e-6)).astype(np.float32)
    wa = []
    for sj in range(1, 21):
        tr = ST[s1[ST] == sj]; te = IDX["source_test"][s1[IDX["source_test"]] == sj]
        perm = np.random.RandomState(sj).permutation(len(tr)); nc = max(4, int(0.25 * len(tr)))
        b = fit_probe(Z[tr[perm[nc:]]], y1[tr[perm[nc:]]], Z[tr[perm[:nc]]], y1[tr[perm[:nc]]], K6, seed)
        with torch.no_grad(): pr = b(torch.from_numpy(Z[te])).numpy().argmax(1)
        wa.append(float((pr == y1[te]).mean()))
    res[seed] = wa; print(seed, np.mean(wa), np.std(wa))
pickle.dump(res, open(os.path.join(RES, "v3_within.pkl"), "wb"))


In [ ]:
%%writefile /content/emg_code/v3_ci.py
"""Participant-clustered bootstrap intervals (R1.11), from the saved main-split encoders."""
import sys, os
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
RES = os.environ.get("EMG_RES", os.path.join(HERE, "results"))
import pickle
from core import *
from lib3 import fit_probe, pstats
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
OUT = os.path.join(RES, "v3_a")
SEEDS = [int(s) for s in os.environ.get("SEEDS", "42,43,44,45,46,47,48,49,50,51").split(",")]
NB = int(os.environ.get("NBOOT", 1000))
st3, tg3 = sub3("source_test"), sub3("target")
per_seed = []
for seed in SEEDS:
    enc = EMGEncoder().to(DEV)
    enc.load_state_dict(torch.load(f"{OUT}/enc_{seed}.pt", map_location=DEV))
    Zr = embed(enc, norm(X1)); m0, s0 = Zr[ST].mean(0), Zr[ST].std(0) + 1e-6
    Zs = lambda Xin: ((embed(enc, Xin) - m0) / s0).astype(np.float32); Z = ((Zr - m0) / s0).astype(np.float32)
    tr3 = sub3("source_train"); ic = np.isin(s1[tr3], CALIB_SUBJECTS)
    b3 = fit_probe(Z[tr3[~ic]], ym[tr3[~ic]], Z[tr3[ic]], ym[tr3[ic]], KC, seed)
    g3 = Gate().fit(Z[tr3[~ic]], ym[tr3[~ic]], KC)
    r = {}
    for nm, Zq, yq in [("source_test", Z[st3], ym[st3]), ("cross-participant", Z[tg3], ym[tg3]),
                       ("cross-dataset", Zs(norm(np.abs(X2))), y2), ("mismatch", Zs(norm(X2)), y2)]:
        p, sc = pstats(b3, g3, Zq); r[nm] = dict(pred=p.argmax(1), y=yq, conf=sc["confidence"], dens=sc["density"])
    per_seed.append(r)
    print("seed", seed, "done", flush=True)
gA_test, gA_tgt, gB = s1[st3], s1[tg3], s2
uT, uG, uB = np.unique(gA_test), np.unique(gA_tgt), np.unique(gB)
ix = lambda g, u, pick: np.concatenate([np.where(g == k)[0] for k in pick])
rng = np.random.RandomState(0)
boot = {k: [] for k in ["bal_cp", "bal_cd", "bal_mm", "auc_conf", "auc_dens"]}
for _ in range(NB):
    iT = ix(gA_test, uT, rng.choice(uT, len(uT))); iG = ix(gA_tgt, uG, rng.choice(uG, len(uG)))
    iB = ix(gB, uB, rng.choice(uB, len(uB)))
    v = {k: [] for k in boot}
    for r in per_seed:
        v["bal_cp"].append(balanced_accuracy_score(r["cross-participant"]["y"][iG], r["cross-participant"]["pred"][iG]))
        v["bal_cd"].append(balanced_accuracy_score(r["cross-dataset"]["y"][iB], r["cross-dataset"]["pred"][iB]))
        v["bal_mm"].append(balanced_accuracy_score(r["mismatch"]["y"][iB], r["mismatch"]["pred"][iB]))
        lab = np.r_[np.zeros(len(iT)), np.ones(len(iB))]
        v["auc_conf"].append(roc_auc_score(lab, np.r_[r["source_test"]["conf"][iT], r["mismatch"]["conf"][iB]]))
        v["auc_dens"].append(roc_auc_score(lab, np.r_[r["source_test"]["dens"][iT], r["mismatch"]["dens"][iB]]))
    for k in boot: boot[k].append(np.mean(v[k]))
CI = {k: tuple(np.percentile(v, [2.5, 97.5]).tolist()) for k, v in boot.items()}
pickle.dump(CI, open(os.path.join(RES, "v3_ci.pkl"), "wb"))
print(CI)


In [ ]:
# Within-participant control (Section IV-L) and participant-clustered bootstrap intervals (Section IV-A, IV-B)
run("v3_within.py", SEEDS=os.environ["SEEDS_WP"])
run("v3_ci.py", SEEDS=os.environ["SEEDS_CI"])


## 6. Aggregation, tables and figures

In [ ]:
%%writefile /content/emg_code/v3_agg.py
"""Aggregate all revised experiments into N.pkl (numbers for text, tables, figures)."""
import sys, glob, pickle, json, numpy as np
import os
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
RES = os.environ.get("EMG_RES", os.path.join(HERE, "results"))
from scipy.stats import ttest_1samp, ttest_rel
from core import aurc
E = RES
ld = lambda pat: [pickle.load(open(f, "rb")) for f in sorted(glob.glob(f"{E}/{pat}"))]
A = ld("v3_a/a_*.pkl"); S = ld("v3_s/s_*.pkl"); import os; NV = ld(os.environ.get("NVD", "v3_n") + "/n_*.pkl"); W = ld(os.environ.get("WD", "v3_w") + "/w_*.pkl")
B = {tc: ld(f"v3_b/{tc}_*.pkl") for tc in ["rectified", "signed", "mixed"]}
LDA = pickle.load(open(f"{E}/v3_lda.pkl", "rb"))
N = dict(nA=len(A), nS=len(S), nNV=len(NV), nW=len(W), nB={k: len(v) for k, v in B.items()})
ms = lambda v: (float(np.nanmean(v)), float(np.nanstd(v)))
mm = lambda v: (float(np.nanmin(v)), float(np.nanmax(v)))
STATS = ["density", "confidence", "entropy", "margin", "energy"]

# ---- ladder (three-class) and six-class
N["mismatch_seed_raw"] = {k: [r["ladder3"]["mismatch"]["auroc"][k] for r in A] for k in ["confidence", "density"]}
N["ladder3"] = {c: {k: ms([r["ladder3"][c][k] for r in A]) for k in ["acc", "bal", "mauc", "ece", "conf", "refusal"]}
                for c in A[0]["ladder3"]}
for c in A[0]["ladder3"]:
    if "auroc" in A[0]["ladder3"][c]:
        N["ladder3"][c]["auroc"] = {k: ms([r["ladder3"][c]["auroc"][k] for r in A]) for k in A[0]["ladder3"][c]["auroc"]}
        N["ladder3"][c]["auroc_range"] = {k: mm([r["ladder3"][c]["auroc"][k] for r in A]) for k in A[0]["ladder3"][c]["auroc"]}
        N["ladder3"][c]["tpr5"] = {k: ms([r["ladder3"][c]["tpr5"][k] for r in A]) for k in STATS}
    for k in ["bal_accepted", "bal_refused", "err_accepted", "err_refused"]:
        if k in A[0]["ladder3"][c]: N["ladder3"][c][k] = ms([r["ladder3"][c].get(k, np.nan) for r in A])
    N["ladder3"][c]["recall"] = np.nanmean([r["ladder3"][c]["recall"] for r in A], 0).tolist()
    N["ladder3"][c]["pred_hist"] = np.mean([r["ladder3"][c]["pred_hist"] for r in A], 0).tolist()
N["six"] = {c: {k: ms([r[c][k] for r in A]) for k in ["acc", "bal", "mauc", "ece", "conf", "refusal",
                                                         "bal_accepted", "bal_refused", "err_accepted", "err_refused"]}
            for c in ["source_test", "target"]}
N["recall6"] = {c: ms([r[c]["recall"] for r in A]) for c in ["source_test", "target"]}
N["recall6"] = {c: (np.mean([r[c]["recall"] for r in A], 0).tolist(), np.std([r[c]["recall"] for r in A], 0).tolist())
                for c in ["source_test", "target"]}
N["T"] = mm([r["T"] for r in A])
N["aurc"] = {k: ms([r["target"]["aurc"][k] for r in A]) for k in STATS}
N["untrained"] = dict(acc=ms([r["untrained"]["acc"] for r in A]),
                      gap=ms([r["untrained"]["aurc"]["density"] - r["untrained"]["aurc"]["confidence"] for r in A]))
N["pre_gap"] = ms([r["target"]["aurc"]["density"] - r["target"]["aurc"]["confidence"] for r in A])
N["latency"] = A[0].get("latency")
pp = {j: [r["per_participant"][j]["acc"] for r in A] for j in range(21, 31)}
N["participant_acc"] = mm([np.mean(v) for v in pp.values()]) + (float(np.std([np.mean(v) for v in pp.values()])),)

# ---- shifts on Dataset A (six-class)
N["shift"] = {}
for c in A[0]["shift"]:
    d = {k: ms([r["shift"][c][k] for r in A]) for k in ["bal", "conf", "refusal"]}
    d["auroc"] = {k: ms([r["shift"][c]["auroc"][k] for r in A]) for k in A[0]["shift"][c]["auroc"]}
    d["tpr5"] = {k: ms([r["shift"][c]["tpr5"][k] for r in A]) for k in STATS}
    if c in LDA["A"]["shift"]:
        d["lda_bal"] = (LDA["A"]["shift"][c]["bal"], 0.0)
        d["lda_auroc"] = {k: (LDA["A"]["shift"][c]["auroc"][k], 0.0) for k in ["confidence", "density"]}
    N["shift"][c] = d
N["lda_source_test_bal6"] = (LDA["A"]["source_test_bal6"], 0.0)
N["lda_ladder3"] = {c: dict(bal=(v["bal"], 0.0), auroc={k: (v["auroc"][k], 0.0) for k in ["confidence", "density"]} if "auroc" in v else None)
                    for c, v in LDA["A"]["ladder3"].items()}

# ---- novelty
N["novelty"] = {k: ms([r["auroc"][k] for r in NV]) for k in STATS}
N["novelty_pairs"] = {str(p): {k: ms([r["auroc"][k] for r in NV if tuple(r["pair"]) == p]) for k in STATS}
                      for p in sorted({tuple(r["pair"]) for r in NV})}

# ---- within B
N["B"] = {}
for tc, rs in B.items():
    if not rs: continue
    N["B"][tc] = {}
    for c in rs[0]["cond"]:
        d = {k: ms([r["cond"][c][k] for r in rs]) for k in ["bal", "acc", "conf", "refusal"]}
        if "auroc" in rs[0]["cond"][c]:
            d["auroc"] = {k: ms([r["cond"][c]["auroc"][k] for r in rs]) for k in rs[0]["cond"][c]["auroc"]}
            d["auroc_range"] = {k: mm([r["cond"][c]["auroc"][k] for r in rs]) for k in rs[0]["cond"][c]["auroc"]}
            d["auroc_raw"] = {k: [r["cond"][c]["auroc"][k] for r in rs] for k in ["confidence", "density"]}
            d["n_below"] = {k: int(sum(r["cond"][c]["auroc"][k] < 0.5 for r in rs)) for k in rs[0]["cond"][c]["auroc"]}
        ob = LDA["B"][tc]
        if ob and c in ob[0]:
            d["lda_bal"] = ms([o[c]["bal"] for o in ob])
            if "auroc" in ob[0][c]:
                d["lda_auroc"] = {k: ms([o[c]["auroc"][k] for o in ob]) for k in ["confidence", "density"]}
        N["B"][tc][c] = d
    N["B"][tc]["_counts"] = np.mean([r["counts_test"] for r in rs], 0).tolist()

# ---- splits
if S:
    N["splits"] = {}
    for c in ["cross-dataset", "mismatch"]:
        N["splits"][c] = dict(bal=ms([r[c]["bal"] for r in S]), conf=ms([r[c]["conf"] for r in S]),
                              auroc={k: ms([r[c]["auroc"][k] for r in S]) for k in STATS},
                              auroc_range={k: mm([r[c]["auroc"][k] for r in S]) for k in STATS},
                              auroc_raw={k: [r[c]["auroc"][k] for r in S] for k in STATS})
    N["splits"]["acc"] = ms([r["acc"] for r in S]); N["splits"]["bal"] = ms([r["bal"] for r in S])
    taps = ["proj", "layer1", "layer2", "layer3_norm"]
    N["depth"] = {t: dict(bal=ms([r["depth"][t]["bal"] for r in S]), acc=ms([r["depth"][t]["acc"] for r in S]),
                          ood=ms([r["depth"][t]["mean_ood"] for r in S]), aurc=ms([r["depth"][t]["aurc_conf"] for r in S]),
                          fault={f: ms([r["depth"][t]["ood"][f] for r in S]) for f in S[0]["depth"][t]["ood"]}) for t in taps}
    N["depth_win"] = int(sum(r["depth"]["layer2"]["mean_ood"] > r["depth"]["layer3_norm"]["mean_ood"] for r in S))

# ---- 200 ms
if W:
    N["w200"] = {c: dict(bal=ms([r["cond"][c]["bal"] for r in W]), conf=ms([r["cond"][c]["conf"] for r in W]),
                         acc=ms([r["cond"][c]["acc"] for r in W]),
                         auroc={k: ms([r["cond"][c]["auroc"][k] for r in W]) for k in STATS} if "auroc" in W[0]["cond"][c] else None,
                         auroc_range={k: mm([r["cond"][c]["auroc"][k] for r in W]) for k in STATS} if "auroc" in W[0]["cond"][c] else None)
                 for c in W[0]["cond"]}
    N["w200_n"] = W[0]["n_windows"]

# ---- cascade
N["cascade"] = {k: {m: ms([r["cascade"][k][m] for r in A]) for m in A[0]["cascade"][k]} for k in A[0]["cascade"]}
N["cascade_meta"] = {k: ms([r["cascade_meta"][k] for r in A]) for k in A[0]["cascade_meta"]}

# ---- statistics
tests = {}
if S:
    v = [r["mismatch"]["auroc"]["confidence"] for r in S]; tests["Confidence inverted under mismatch, A to B"] = ttest_1samp(v, 0.5).pvalue
    v = [r["mismatch"]["auroc"]["density"] for r in S]; tests["Density detects mismatch, A to B"] = ttest_1samp(v, 0.5).pvalue
    v = [r["cross-dataset"]["bal"] for r in S]; tests["Cross-dataset balanced acc. above chance"] = ttest_1samp(v, 1 / 3).pvalue
    a = [r["depth"]["layer2"]["mean_ood"] for r in S]; b = [r["depth"]["layer3_norm"]["mean_ood"] for r in S]
    tests["Detection, block 2 vs output"] = ttest_rel(a, b).pvalue
    a = [r["depth"]["layer3_norm"]["bal"] for r in S]; b = [r["depth"]["proj"]["bal"] for r in S]
    tests["Balanced acc., output vs projection"] = ttest_rel(a, b).pvalue
if B["rectified"]:
    v = [r["cond"]["signed (no rectification)"]["auroc"]["confidence"] for r in B["rectified"]]
    tests["Confidence inverted under mismatch, within B"] = ttest_1samp(v, 0.5).pvalue
    v = [r["cond"]["signed (no rectification)"]["auroc"]["density"] for r in B["rectified"]]
    tests["Density detects mismatch, within B"] = ttest_1samp(v, 0.5).pvalue
a = [r["cascade"]["range + density + confidence"]["harmful"] for r in A]; b = [r["cascade"]["confidence"]["harmful"] for r in A]
tests["Harmful acceptance, layered vs confidence"] = ttest_rel(a, b).pvalue
# participant-clustered bootstrap for error ranking, averaged over seeds
rng = np.random.RandomState(0); parts = A[0]["target"]["part"] if "part" in A[0]["target"] else None
import core
TG = core.IDX["target"]; part = core.s1[TG]; uniq = np.unique(part); idxs = {u: np.where(part == u)[0] for u in uniq}
diffs = {k: [] for k in ["density", "entropy", "margin", "energy"]}
for _ in range(2000):
    pick = rng.choice(uniq, len(uniq), replace=True); ix = np.concatenate([idxs[u] for u in pick])
    acc = {k: 0.0 for k in diffs}
    for r in A:
        sc = r["target"]["scores"]; cr = r["target"]["correct"]
        ref = aurc(sc["confidence"][ix], cr[ix])
        for k in diffs: acc[k] += aurc(sc[k][ix], cr[ix]) - ref
    for k in diffs: diffs[k].append(acc[k] / len(A))
N["boot"] = {}
for k, dd in diffs.items():
    dd = np.array(dd); lo, md, hi = np.percentile(dd, [2.5, 50, 97.5])
    p = max(2 * min((dd <= 0).mean(), (dd >= 0).mean()), 1 / 2001)
    N["boot"][k] = dict(d=float(md), lo=float(lo), hi=float(hi), p=float(p))
    tests[f"{k.capitalize()} vs max softmax, error ranking"] = p
names = list(tests); p = np.array([tests[k] for k in names]); m = len(p)
o = np.argsort(p); q = np.empty(m); prev = 1.0
for rank, i in reversed(list(enumerate(o, 1))):
    prev = min(prev, p[i] * m / rank); q[i] = prev
N["tests"] = [dict(name=names[i], p=float(p[i]), bonf=float(min(1, p[i] * m)), q=float(q[i])) for i in o]
pickle.dump(N, open(f"{E}/N.pkl", "wb"))
print(json.dumps({k: N[k] for k in ["nA", "nS", "nNV", "nW", "nB"]}))
for t in N["tests"]: print(f"  {t['name']:48s} p={t['p']:.2e} q={t['q']:.2e} bonf={t['bonf']:.2e}")
# per-participant cross-participant results (Table XIII)
pp = {j: {k: ms([r["per_participant"][j][k] for r in A]) for k in ["acc", "bal", "conf"]} for j in range(21, 31)}
pickle.dump(pp, open(f"{E}/v3_pp.pkl", "wb"))


In [ ]:
%%writefile /content/emg_code/v3_tables.py
"""Write Tables III to XII of the manuscript as CSV files and print the headline numbers."""
import sys, os, pickle, numpy as np, pandas as pd
HERE = os.path.dirname(os.path.abspath(__file__)); sys.path.insert(0, HERE)
RES = os.environ.get("EMG_RES", os.path.join(HERE, "results"))
N = pickle.load(open(os.path.join(RES, "N.pkl"), "rb"))
OUT = os.path.join(RES, "tables"); os.makedirs(OUT, exist_ok=True)
f3 = lambda t: f"{t[0]:.3f} ± {t[1]:.3f}"; g3 = lambda t: f"{t[0]:.3f}"
L, S, B, C, D = N["ladder3"], N["shift"], N["B"], N["cascade"], N["depth"]
RB, SB, XB = B["rectified"], B["signed"], B["mixed"]
def save(name, cols, rows):
    pd.DataFrame(rows, columns=cols).to_csv(os.path.join(OUT, name + ".csv"), index=False)

gap = lambda d: f"{100 * (d['bal_accepted'][0] - d['bal_refused'][0]):.1f}"
save("table03_ladder", ["Partition", "Pooled acc.", "Balanced acc.", "Refusal", "Mean conf.", "Refused gap (points)"],
     [[n, g3(L[c]["acc"]), f3(L[c]["bal"]), g3(L[c]["refusal"]), g3(L[c]["conf"]), gap(L[c])] for n, c in
      [("Source test", "source_test"), ("Cross-participant", "cross-participant"), ("Cross-dataset", "cross-dataset"),
       ("Conditioning mismatch", "mismatch")]] +
     [[f"Six-class {n}", g3(N["six"][c]["acc"]), f3(N["six"][c]["bal"]), g3(N["six"][c]["refusal"]), g3(N["six"][c]["conf"]), gap(N["six"][c])]
      for n, c in [("source test", "source_test"), ("cross-participant", "target")]])
a4 = lambda d: [g3(d["density"]), g3(d["confidence"]), g3(d["energy"])]
rows = [["Held-out gesture"] + a4(N["novelty"]) + ["n/a"]]
for f in ["channel rotation", "gain shift", "channel dropout", "broadband noise", "motion artefact"]:
    rows.append([f.replace("artefact", "artifact").capitalize()] + a4(S[f]["auroc"]) + [g3(S[f]["auroc"]["range check"])])
for c, n in [("cross-participant", "New participant"), ("cross-dataset", "Cross-dataset"), ("mismatch", "Conditioning mismatch")]:
    rows.append([n] + a4(L[c]["auroc"]) + [g3(L[c]["auroc"]["range check"])])
save("table04_detection", ["Condition", "Density", "Max softmax", "Energy", "Range check"], rows)
rb = lambda tr, te, d: [tr, te, f3(d["bal"]), g3(d["conf"])] + ([g3(d["auroc"][k]) for k in ["confidence", "density", "range check"]] if "auroc" in d else ["ref."] * 3)
save("table05_within_B", ["Trained on", "Tested on", "Balanced acc.", "Mean conf.", "Max softmax", "Density", "Range check"],
     [rb("rectified", "rectified", RB["matched"]), rb("rectified", "signed", RB["signed (no rectification)"]),
      rb("signed", "signed", SB["matched"]), rb("signed", "rectified", SB["rectified"]),
      rb("both", "rectified", XB["matched"]), rb("both", "signed", XB["signed (no rectification)"]), rb("both", "gain x0.1", XB["gain x0.1"])])
fam = lambda d: [f3(d["bal"]), g3(d["auroc"]["confidence"]), g3(d["auroc"]["density"]), g3(d["auroc"]["range check"])]
save("table06_mismatch_family", ["Dataset", "Mismatch", "Balanced acc.", "Max softmax", "Density", "Range check"],
     [["A", c] + fam(S[c]) for c in ["gain x0.1", "gain x0.25", "gain x0.5", "gain x2", "gain x4", "gain x10", "envelope 150 ms",
                                       "per-window z-score", "400 Hz stream"]] +
     [["B", c] + fam(RB[c]) for c in ["gain x0.1", "gain x4", "envelope 150 ms", "per-window z-score", "400 Hz stream", "band-pass 20-90 Hz"]])
tests = {t["name"]: t for t in N["tests"]}
save("fig6_error_ranking_values", ["Statistic", "AURC", "Bootstrap difference vs max softmax", "BH q"],
     [[k, f"{N['aurc'][k][0]:.4f} ± {N['aurc'][k][1]:.4f}", "" if k == "confidence" else f"{N['boot'][k]['d']:.4f} [{N['boot'][k]['lo']:.4f}, {N['boot'][k]['hi']:.4f}]",
       "" if k == "confidence" else f"{tests[k.capitalize() + ' vs max softmax, error ranking']['q']:.4f}"] for k in ["confidence", "entropy", "margin", "density", "energy"]])
lr = N["lda_ladder3"]; lw, lwm = RB["matched"], RB["signed (no rectification)"]
save("table07_classical_lda", ["Condition", "Learned bal.", "LDA bal.", "Learned max softmax AUROC", "LDA max softmax AUROC", "LDA density AUROC"],
     [["A source test (6)", g3(N["six"]["source_test"]["bal"]), g3(N["lda_source_test_bal6"]), "ref.", "ref.", "ref."]] +
     [[f"A {c} (6)", g3(S[c]["bal"]), g3(S[c]["lda_bal"]), g3(S[c]["auroc"]["confidence"]), g3(S[c]["lda_auroc"]["confidence"]), g3(S[c]["lda_auroc"]["density"])]
      for c in ["gain x0.1", "envelope 150 ms"]] +
     [[f"A to B {n} (3)", g3(L[c]["bal"]), g3(lr[c]["bal"]), g3(L[c]["auroc"]["confidence"]), g3(lr[c]["auroc"]["confidence"]), g3(lr[c]["auroc"]["density"])]
      for n, c in [("rectified", "cross-dataset"), ("signed", "mismatch")]] +
     [["Within B matched (3)", g3(lw["bal"]), g3(lw["lda_bal"]), "ref.", "ref.", "ref."],
      ["Within B signed (3)", g3(lwm["bal"]), g3(lwm["lda_bal"]), g3(lwm["auroc"]["confidence"]), g3(lwm["lda_auroc"]["confidence"]), g3(lwm["lda_auroc"]["density"])]])
save("table08_layered_checks", ["Gate", "Clean accepted %", "Faulty accepted %", "Balanced acc. accepted", "Harmful %"],
     [[k, f"{100 * v['cov_clean'][0]:.1f}", f"{100 * v['cov_faulty'][0]:.1f}", g3(v["bal_accepted"]), f"{100 * v['harmful'][0]:.1f} ± {100 * v['harmful'][1]:.1f}"] for k, v in C.items()])
save("table09_depth", ["Tap", "Balanced acc.", "Mean fault detection AUROC", "AURC"],
     [[t, f3(D[t]["bal"]), f"{D[t]['ood'][0]:.4f} ± {D[t]['ood'][1]:.4f}", f"{D[t]['aurc'][0]:.4f} ± {D[t]['aurc'][1]:.4f}"] for t in D])
cls = ["rest", "fist", "open palm", "wrist left", "wrist right", "thumbs-up"]
(rs, ss), (rt, st) = N["recall6"]["source_test"], N["recall6"]["target"]
save("table10_per_class", ["Class", "Recall source test", "Recall cross-participant", "Drop"],
     [[cls[i], f"{rs[i]:.3f} ± {ss[i]:.3f}", f"{rt[i]:.3f} ± {st[i]:.3f}", f"{rs[i] - rt[i]:.3f}"] for i in np.argsort(np.array(rs) - np.array(rt))])
pp = pickle.load(open(os.path.join(RES, "v3_pp.pkl"), "rb"))
save("table11_per_participant", ["Participant", "Accuracy", "Balanced acc.", "Mean conf."],
     [[j, f3(v["acc"]), f3(v["bal"]), f3(v["conf"])] for j, v in sorted(pp.items())])
save("table12_tests", ["Test", "p", "Bonferroni", "BH q", "Survives"],
     [[t["name"], f"{t['p']:.2e}", f"{t['bonf']:.2e}", f"{t['q']:.2e}", t["q"] < 0.05] for t in N["tests"]])
MM = L["mismatch"]
print("Headline numbers (values reported in the paper in brackets):")
print(f"  mismatch balanced acc.            {f3(MM['bal'])}   [0.342 ± 0.016]")
print(f"  mismatch max-softmax AUROC        {f3(MM['auroc']['confidence'])}   [0.179 ± 0.093]")
print(f"  mismatch density AUROC            {f3(MM['auroc']['density'])}   [0.671 ± 0.013]")
print(f"  within B, rect->signed MSP AUROC  {f3(RB['signed (no rectification)']['auroc']['confidence'])}   [0.089 ± 0.131]")
print(f"  gain x0.1 MSP / density AUROC     {g3(S['gain x0.1']['auroc']['confidence'])} / {g3(S['gain x0.1']['auroc']['density'])}   [0.174 / 0.378]")
print(f"  harmful acceptance, none -> all   {100 * C['none']['harmful'][0]:.1f}% -> {100 * C['range + density + confidence']['harmful'][0]:.1f}%   [22.9% -> 3.3%]")
print(f"  tables written to {OUT}")


In [ ]:
%%writefile /content/emg_code/v3_figs.py
"""Figures for the revised manuscript, single-column width, from N.pkl."""
import pickle, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import os
RES = os.environ.get("EMG_RES", os.path.join(os.path.dirname(os.path.abspath(__file__)), "results"))
N = pickle.load(open(os.path.join(RES, "N.pkl"), "rb"))
OUT = os.path.join(RES, "figures"); os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["DejaVu Serif"],
    "font.size": 7, "axes.labelsize": 7, "axes.titlesize": 7,
    "xtick.labelsize": 6, "ytick.labelsize": 6, "legend.fontsize": 6,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.dpi": 300, "savefig.dpi": 300, "savefig.bbox": "tight"})
CD, CC, CN, CB, CO = "#b02418", "#1a7a4c", "#6b6b6b", "#2a5d9c", "#c98a00"
W = 3.4

# ---- Fig. 2: ladder ------------------------------------------------------------
L = N["ladder3"]; conds = ["source_test", "cross-participant", "cross-dataset", "mismatch"]
lbl = ["source\ntest", "cross-\nparticipant", "cross-\ndataset", "conditioning\nmismatch"]
x = np.arange(4)
fig, ax = plt.subplots(3, 1, figsize=(W, 4.6))
ax[0].errorbar(x, [L[c]["acc"][0] for c in conds], [L[c]["acc"][1] for c in conds], marker="o", ms=3.5, ls="--", color=CN, capsize=2, label="pooled accuracy")
ax[0].errorbar(x, [L[c]["bal"][0] for c in conds], [L[c]["bal"][1] for c in conds], marker="s", ms=3.5, color=CD, capsize=2, label="balanced accuracy")
ax[0].axhline(1/3, color="k", ls=":", lw=0.7); ax[0].set_ylabel("accuracy"); ax[0].set_title("(a) pooled accuracy conceals the collapse")
ax[0].legend(loc="lower left", frameon=False)
ax[1].errorbar(x, [L[c]["conf"][0] for c in conds], [L[c]["conf"][1] for c in conds], marker="o", ms=3.5, color=CC, capsize=2, label="mean confidence")
ax[1].errorbar(x, [L[c]["bal"][0] for c in conds], [L[c]["bal"][1] for c in conds], marker="s", ms=3.5, color=CD, capsize=2, label="balanced accuracy")
ax[1].set_ylabel("value"); ax[1].set_title("(b) confidence rises as accuracy falls"); ax[1].legend(loc="lower left", frameon=False)
ax[2].bar(x, [L[c]["refusal"][0] * 100 for c in conds], 0.6, yerr=[L[c]["refusal"][1] * 100 for c in conds], color="#8fa9c9", edgecolor="#333", lw=0.4, capsize=2)
ax[2].set_ylabel("refusal rate (%)"); ax[2].set_title("(c) refusal is not monotone in severity")
for a in ax: a.set_xticks(x); a.set_xticklabels(lbl, fontsize=5.8); a.grid(alpha=0.25, axis="y")
fig.tight_layout(h_pad=0.8); fig.savefig(f"{OUT}/fig2.png"); plt.close(fig)

# ---- Fig. 3: inversion across runs ------------------------------------------------
B = N["B"]
groups = [("A to B, 10 seeds", None), ("A to B, 10 splits", N["splits"]["mismatch"]["auroc_raw"]),
          ("within B, rectified\nto signed", B["rectified"]["signed (no rectification)"]["auroc_raw"]),
          ("within B, signed\nto rectified", B["signed"]["rectified"]["auroc_raw"])]
seedv = N["mismatch_seed_raw"]
groups[0] = (groups[0][0], seedv)
fig, ax = plt.subplots(figsize=(W, 2.3)); rng = np.random.RandomState(0)
for i, (nm, d) in enumerate(groups):
    y = len(groups) - 1 - i
    for k, col, off, mk in [("confidence", CC, 0.14, "o"), ("density", CD, -0.14, "s")]:
        v = np.array(d[k]); ax.scatter(v, y + off + rng.uniform(-0.05, 0.05, len(v)), s=9, color=col, marker=mk,
                                       edgecolor="#222", lw=0.3, label=("max softmax" if k == "confidence" else "density") if i == 0 else None, zorder=3)
        ax.plot([v.mean()] * 2, [y + off - 0.1, y + off + 0.1], color=col, lw=1.4, zorder=4)
ax.axvline(0.5, color="k", ls="--", lw=0.7); ax.set_xlim(0, 1)
ax.set_yticks(range(len(groups))); ax.set_yticklabels([g[0] for g in groups][::-1], fontsize=6)
ax.set_xlabel("detection AUROC (mismatched vs. matched input)"); ax.grid(alpha=0.25, axis="x")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False, fontsize=5.8)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3.png"); plt.close(fig)

# ---- Fig. 4: mismatch family on A ------------------------------------------------
S = N["shift"]
fam = ["gain x0.1", "gain x0.25", "gain x0.5", "gain x2", "gain x4", "gain x10", "envelope 50 ms", "envelope 150 ms",
       "per-window z-score", "400 Hz stream"]
nice = ["gain ×0.1", "gain ×0.25", "gain ×0.5", "gain ×2", "gain ×4", "gain ×10", "envelope 50 ms", "envelope 150 ms",
        "per-window std.", "400 Hz stream"]
y = np.arange(len(fam))[::-1]
fig, ax = plt.subplots(2, 1, figsize=(W, 4.4), gridspec_kw=dict(height_ratios=[1, 1.25]))
ax[0].barh(y, [S[c]["bal"][0] for c in fam], 0.6, xerr=[S[c]["bal"][1] for c in fam], color="#8fa9c9", edgecolor="#333", lw=0.4, capsize=1.5)
ax[0].axvline(S and N["six"]["source_test"]["bal"][0], color=CB, ls="-", lw=0.8)
ax[0].axvline(1/6, color="k", ls=":", lw=0.7)
ax[0].set_yticks(y); ax[0].set_yticklabels(nice, fontsize=5.8); ax[0].set_xlim(0, 1)
ax[0].set_xlabel("balanced accuracy (six classes)"); ax[0].set_title("(a) accuracy under each mismatch")
h = 0.26
for k, col, off, nm in [("confidence", CC, h, "max softmax"), ("density", CD, 0, "density"), ("range check", CN, -h, "range check")]:
    ax[1].barh(y + off, [S[c]["auroc"][k][0] for c in fam], h, color=col, edgecolor="#333", lw=0.3, label=nm)
ax[1].axvline(0.5, color="k", ls="--", lw=0.7); ax[1].set_xlim(0, 1)
ax[1].set_yticks(y); ax[1].set_yticklabels(nice, fontsize=5.8); ax[1].set_xlabel("detection AUROC")
ax[1].set_title("(b) detection of each mismatch"); ax[1].legend(loc="upper center", bbox_to_anchor=(0.42, -0.2), ncol=3, frameon=False, fontsize=5.6)
for a in ax: a.grid(alpha=0.25, axis="x")
fig.tight_layout(h_pad=0.8); fig.savefig(f"{OUT}/fig4.png"); plt.close(fig)

# ---- Fig. 5: fault suite detection ----------------------------------------------
rows = [("held-out gesture", N["novelty"])] + [(n.replace("artefact", "artifact"), S[n]["auroc"]) for n in
        ["channel rotation", "gain shift", "channel dropout", "broadband noise", "motion artefact"]] + \
       [("new participant", L["cross-participant"]["auroc"]), ("cross-dataset", L["cross-dataset"]["auroc"]),
        ("conditioning mismatch", L["mismatch"]["auroc"])]
y = np.arange(len(rows))[::-1]; h = 0.36
fig, ax = plt.subplots(figsize=(W, 2.9))
ax.barh(y + h/2, [r[1]["density"][0] for r in rows], h, xerr=[r[1]["density"][1] for r in rows], color=CD, edgecolor="#333", lw=0.35, capsize=1.2, label="density")
ax.barh(y - h/2, [r[1]["confidence"][0] for r in rows], h, xerr=[r[1]["confidence"][1] for r in rows], color=CC, edgecolor="#333", lw=0.35, capsize=1.2, label="max softmax")
ax.axvline(0.5, color="k", ls="--", lw=0.8); ax.axhspan(-0.5, 2.5, color="#f2f2f2", zorder=0)
ax.set_yticks(y); ax.set_yticklabels([r[0] for r in rows], fontsize=6); ax.set_xlim(0, 1)
ax.set_xlabel("detection AUROC"); ax.legend(loc="lower right", frameon=True, framealpha=0.9, fontsize=5.8); ax.grid(alpha=0.25, axis="x")
fig.tight_layout(); fig.savefig(f"{OUT}/fig5.png"); plt.close(fig)

# ---- Fig. 6: error ranking -------------------------------------------------------
st = ["confidence", "entropy", "margin", "density", "energy"]; nm = ["max softmax", "entropy", "margin", "density", "energy"]
fig, ax = plt.subplots(2, 1, figsize=(W, 3.6))
ax[0].bar(range(5), [N["aurc"][k][0] for k in st], 0.6, yerr=[N["aurc"][k][1] for k in st], capsize=1.8,
          color=[CC, CC, CC, CD, CO], edgecolor="#333", lw=0.35)
ax[0].set_xticks(range(5)); ax[0].set_xticklabels(nm, fontsize=6); ax[0].set_ylabel("AURC (lower is better)")
ax[0].set_title("(a) error ranking across participants"); ax[0].grid(alpha=0.25, axis="y")
bk = ["entropy", "margin", "density", "energy"]; bo = N["boot"]
ax[1].errorbar([bo[k]["d"] for k in bk], range(4)[::-1], xerr=[[bo[k]["d"] - bo[k]["lo"] for k in bk], [bo[k]["hi"] - bo[k]["d"] for k in bk]],
               fmt="o", ms=3.5, color="#333", capsize=2)
ax[1].axvline(0, color="k", ls="--", lw=0.7); ax[1].set_yticks(range(4)[::-1]); ax[1].set_yticklabels(bk, fontsize=6)
ax[1].set_xlabel("difference in AURC vs. max softmax (95% interval)"); ax[1].set_title("(b) participant-clustered bootstrap")
ax[1].grid(alpha=0.25, axis="x")
fig.tight_layout(h_pad=0.8); fig.savefig(f"{OUT}/fig6.png"); plt.close(fig)

# ---- Fig. 7: cascade -------------------------------------------------------------
C = N["cascade"]; pol = ["none", "confidence", "density", "range check", "range + density", "range + density + confidence"]
pn = ["no gate", "max\nsoftmax", "density", "range\ncheck", "range +\ndensity", "all three"]
fig, ax = plt.subplots(2, 1, figsize=(W, 3.7))
x = np.arange(len(pol))
ax[0].bar(x, [C[p]["harmful"][0] * 100 for p in pol], 0.6, yerr=[C[p]["harmful"][1] * 100 for p in pol], color=CD, edgecolor="#333", lw=0.35, capsize=1.8)
ax[0].set_ylabel("harmful acceptance (%)"); ax[0].set_title("(a) faulty windows accepted and misclassified")
w = 0.38
ax[1].bar(x - w/2, [C[p]["cov_clean"][0] * 100 for p in pol], w, color=CB, edgecolor="#333", lw=0.35, label="clean windows")
ax[1].bar(x + w/2, [C[p]["cov_faulty"][0] * 100 for p in pol], w, color="#d9a29a", edgecolor="#333", lw=0.35, label="faulty windows")
ax[1].set_ylabel("accepted (%)"); ax[1].set_title("(b) coverage"); ax[1].legend(loc="lower left", frameon=True, framealpha=0.9, fontsize=5.6)
for a in ax: a.set_xticks(x); a.set_xticklabels(pn, fontsize=5.4); a.grid(alpha=0.25, axis="y")
fig.tight_layout(h_pad=0.8); fig.savefig(f"{OUT}/fig7.png"); plt.close(fig)

# ---- Fig. 8: 200 ms and factorial ------------------------------------------------
w2 = N["w200"]; x = np.arange(4)
fig, ax = plt.subplots(3, 1, figsize=(W, 4.8))
ax[0].errorbar(x, [w2[c]["bal"][0] for c in conds], [w2[c]["bal"][1] for c in conds], marker="s", ms=3.5, color=CD, capsize=2, label="balanced accuracy")
ax[0].errorbar(x, [w2[c]["conf"][0] for c in conds], [w2[c]["conf"][1] for c in conds], marker="o", ms=3.5, color=CC, capsize=2, label="mean confidence")
ax[0].axhline(1/3, color="k", ls=":", lw=0.7); ax[0].set_xticks(x); ax[0].set_xticklabels(lbl, fontsize=5.8)
ax[0].set_title("(a) shift ladder with 200 ms windows"); ax[0].legend(loc="lower left", frameon=False)
c3 = ["cross-participant", "cross-dataset", "mismatch"]; x3 = np.arange(3); h = 0.36
ax[1].bar(x3 - h/2, [w2[c]["auroc"]["density"][0] for c in c3], h, yerr=[w2[c]["auroc"]["density"][1] for c in c3], color=CD, edgecolor="#333", lw=0.35, capsize=1.5, label="density")
ax[1].bar(x3 + h/2, [w2[c]["auroc"]["confidence"][0] for c in c3], h, yerr=[w2[c]["auroc"]["confidence"][1] for c in c3], color=CC, edgecolor="#333", lw=0.35, capsize=1.5, label="max softmax")
ax[1].axhline(0.5, color="k", ls="--", lw=0.7); ax[1].set_xticks(x3); ax[1].set_xticklabels(["cross-participant", "cross-dataset", "mismatch"], fontsize=5.8)
ax[1].set_ylim(0, 1); ax[1].set_ylabel("detection AUROC"); ax[1].set_title("(b) detection with 200 ms windows"); ax[1].legend(loc="upper left", frameon=False)
fc = ["cross-dataset", "mismatch", "signed + own norm", "rectified + own norm"]
fl = ["rectified,\nsource stats", "signed,\nsource stats", "signed,\nown stats", "rectified,\nown stats"]
x4 = np.arange(4)
ax[2].bar(x4 - 0.27, [L[c]["bal"][0] for c in fc], 0.27, color="#8fa9c9", edgecolor="#333", lw=0.35, label="balanced acc.")
ax[2].bar(x4, [L[c]["auroc"]["confidence"][0] for c in fc], 0.27, color=CC, edgecolor="#333", lw=0.35, label="max softmax AUROC")
ax[2].bar(x4 + 0.27, [L[c]["auroc"]["density"][0] for c in fc], 0.27, color=CD, edgecolor="#333", lw=0.35, label="density AUROC")
ax[2].axhline(0.5, color="k", ls="--", lw=0.7); ax[2].set_xticks(x4); ax[2].set_xticklabels(fl, fontsize=5.6); ax[2].set_ylim(0, 1.05)
ax[2].set_title("(c) conditioning by normalization, Dataset B input"); ax[2].legend(loc="upper center", bbox_to_anchor=(0.5, -0.3), frameon=False, fontsize=5.4, ncol=3)
for a in ax: a.grid(alpha=0.25, axis="y")
fig.tight_layout(h_pad=0.8); fig.savefig(f"{OUT}/fig8.png"); plt.close(fig)

# ---- Fig. 9 and 10: depth ----------------------------------------------------------
D = N["depth"]; taps = ["proj", "layer1", "layer2", "layer3_norm"]; tl = ["patch\nprojection", "block 1", "block 2", "block 3\n(output)"]
x = np.arange(4)
fig, ax = plt.subplots(figsize=(W, 2.3))
l1, = ax.plot(x, [D[t]["ood"][0] for t in taps], "-o", ms=3.5, color=CD, label="mean fault detection AUROC")
l2, = ax.plot(x, [D[t]["bal"][0] for t in taps], "-s", ms=3.5, color=CB, label="balanced accuracy")
ax.set_ylim(0.55, 0.75); ax.set_ylabel("AUROC / balanced accuracy"); ax.set_xticks(x); ax.set_xticklabels(tl, fontsize=6)
ax.grid(alpha=0.25, axis="y")
ax2 = ax.twinx()
l3, = ax2.plot(x, [D[t]["aurc"][0] for t in taps], "--^", ms=3.5, color=CC, label="AURC of max softmax (right axis)")
ax2.set_ylim(0.050, 0.065); ax2.set_ylabel("AURC (lower is better)"); ax2.tick_params(labelsize=6)
ax.legend(handles=[l1, l2, l3], loc="upper center", bbox_to_anchor=(0.5, -0.24), ncol=2, frameon=False)
fig.tight_layout(); fig.savefig(f"{OUT}/fig9.png"); plt.close(fig)
faults = list(D["proj"]["fault"]); fn = [f.replace("artefact", "artifact") for f in faults]
fig, ax = plt.subplots(figsize=(W, 2.4)); cols = [CD, CB, CC, CO, CN]
for f, n_, c in zip(faults, fn, cols):
    ax.errorbar(x, [D[tp]["fault"][f][0] for tp in taps], [D[tp]["fault"][f][1] for tp in taps], marker="o", ms=3, color=c, capsize=1.5, label=n_)
ax.axhline(0.5, color="k", ls="--", lw=0.7); ax.set_xticks(x); ax.set_xticklabels(tl, fontsize=5.8)
ax.set_ylabel("density detection AUROC"); ax.grid(alpha=0.25, axis="y")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=3, frameon=False, fontsize=5.6)
fig.tight_layout(); fig.savefig(f"{OUT}/fig10.png"); plt.close(fig)
print("figures written")


In [ ]:
run("v3_agg.py")
run("v3_tables.py")
run("v3_figs.py")


In [ ]:
# Show the tables and figures
import pandas as pd
from IPython.display import display, Image, Markdown
RES = os.environ["EMG_RES"]
for f in sorted(glob.glob(os.path.join(RES, "tables", "*.csv"))):
    display(Markdown("**" + os.path.basename(f) + "**")); display(pd.read_csv(f))
for f in sorted(glob.glob(os.path.join(RES, "figures", "*.png")), key=lambda p: int(os.path.basename(p)[3:-4])):
    display(Markdown("**" + os.path.basename(f) + "**")); display(Image(f, width=420))


In [ ]:
# Bundle everything and download it to your computer
import shutil
zip_path = shutil.make_archive(os.path.join("/content", "emg_v10_results"), "zip", RES)
print("archive:", zip_path, round(os.path.getsize(zip_path) / 1e6, 1), "MB")
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
